In [1]:
import os

file_path = r"C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\raw\Home_and_Kitchen.train.csv"

print("File exists:", os.path.exists(file_path))

File exists: True


In [2]:
import pandas as pd

df_sample = pd.read_csv(file_path, nrows=10)

print(df_sample)

                        user_id parent_asin  rating      timestamp
0  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B00J10VZ2W     5.0  1523092290368
1  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B09KX3FZQS     1.0  1545115669044
2  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B0CGY43Y3P     3.0  1552877983695
3  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B08K8N5FB2     5.0  1581799806318
4  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B00GUAURXY     5.0  1583557787276
5  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B0C6V27S6N     5.0  1589934525940
6  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B08CSZDXZY     5.0  1651855096178
7  AFKZENTNBQ7A7V7UXW5JJI6UGRYQ  B09G2PW8ZG     2.0  1653447296788
8  AGGZ357AO26RQZVRLGU4D4N52DZQ  B0778KV29D     4.0  1341922513000
9  AGGZ357AO26RQZVRLGU4D4N52DZQ  B002FWQU3W     4.0  1354390111000


In [3]:
import pandas as pd

user_count = set()
product_count = set()
rating_counts = {}
total_rows = 0

for chunk in pd.read_csv(file_path, chunksize=100000):

    total_rows += len(chunk)

    user_count.update(chunk["user_id"])
    product_count.update(chunk["parent_asin"])

    counts = chunk["rating"].value_counts()

    for rating, count in counts.items():
        rating_counts[rating] = rating_counts.get(rating, 0) + count

print("===== DATASET STATISTICS =====")
print("Total interactions:", total_rows)
print("Unique users:", len(user_count))
print("Unique products:", len(product_count))

print("\n===== RATING DISTRIBUTION =====")

for rating in sorted(rating_counts):
    print(f"{rating} stars: {rating_counts[rating]}")

===== DATASET STATISTICS =====
Total interactions: 22362598
Unique users: 2920001
Unique products: 762198

===== RATING DISTRIBUTION =====
1.0 stars: 1403631
2.0 stars: 911905
3.0 stars: 1523943
4.0 stars: 2814219
5.0 stars: 15708900


In [4]:
import pandas as pd

product_counts = {}

for chunk in pd.read_csv(file_path, chunksize=100000):

    counts = chunk["parent_asin"].value_counts()

    for product, count in counts.items():
        product_counts[product] = product_counts.get(product, 0) + count

print("Total unique products:", len(product_counts))

top_products = sorted(
    product_counts.items(),
    key=lambda x: x[1],
    reverse=True
)[:30000]

print("\nTop 10 products:")
for product, count in top_products[:10]:
    print(product, "→", count, "interactions")

Total unique products: 762198

Top 10 products:
B0B53DWRVW → 27744 interactions
B0BSPGR5T6 → 18616 interactions
B00U8QEXBS → 17536 interactions
B0C7LZYYWQ → 17311 interactions
B01NBTD4Q8 → 15814 interactions
B0BZJP9VGZ → 15791 interactions
B0BZJNL66N → 12408 interactions
B08WYH9T9Y → 10573 interactions
B0C5VFZ4GK → 10155 interactions
B08K22FTF1 → 10155 interactions


In [5]:
top_product_ids = set(product for product, count in top_products)

print("Products selected:", len(top_product_ids))

Products selected: 30000


In [6]:
import pandas as pd

top_products_df = pd.DataFrame({
    "parent_asin": list(top_product_ids)
})

output_path = "../data/processed/top_30000_products.csv"

top_products_df.to_csv(output_path, index=False)

print("Saved:", output_path)

Saved: ../data/processed/top_30000_products.csv


In [7]:
import sys

print(sys.executable)

c:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Scripts\python.exe


In [8]:
from datasets import load_dataset

print("datasets is working!")

c:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


ImportError: DLL load failed while importing _xxhash: An Application Control policy has blocked this file.

In [9]:
import glob
import pyarrow.parquet as pq

metadata_path = r"C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\raw\metadata\raw_meta_Home_and_Kitchen"

files = glob.glob(metadata_path + r"\*.parquet")

print("Metadata files found:", len(files))

parquet_file = pq.ParquetFile(files[0])

print("\nFirst file:")
print(files[0])

print("\nColumns:")
print(parquet_file.schema.names)

Metadata files found: 21

First file:
C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\raw\metadata\raw_meta_Home_and_Kitchen\full-00000-of-00021.parquet

Columns:
['main_category', 'title', 'average_rating', 'rating_number', 'element', 'element', 'price', 'element', 'element', 'element', 'element', 'element', 'element', 'element', 'store', 'element', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']


In [10]:
import pandas as pd
import glob
import os

# Path to metadata files
metadata_path = r"C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\raw\metadata\raw_meta_Home_and_Kitchen"

# Path to our selected products
selected_products_path = "../data/processed/top_30000_products.csv"

# Load the 30,000 product IDs
selected_products = set(
    pd.read_csv(selected_products_path)["parent_asin"]
)

print("Selected products:", len(selected_products))

# Find all metadata files
metadata_files = glob.glob(metadata_path + r"\*.parquet")

print("Metadata files:", len(metadata_files))

matched_products = []

# Process one Parquet file at a time
for i, file in enumerate(metadata_files):

    print(f"\nProcessing file {i+1}/{len(metadata_files)}")

    df = pd.read_parquet(file)

    # Keep only products we selected
    matched = df[df["parent_asin"].isin(selected_products)]

    print("Matches found:", len(matched))

    if len(matched) > 0:
        matched_products.append(matched)

print("\nCombining matched products...")

products_df = pd.concat(
    matched_products,
    ignore_index=True
)

print("Total matched products:", len(products_df))

Selected products: 30000
Metadata files: 21

Processing file 1/21
Matches found: 6423

Processing file 2/21
Matches found: 3643

Processing file 3/21
Matches found: 2654

Processing file 4/21
Matches found: 2232

Processing file 5/21
Matches found: 1796

Processing file 6/21
Matches found: 1600

Processing file 7/21
Matches found: 1427

Processing file 8/21
Matches found: 1203

Processing file 9/21
Matches found: 1197

Processing file 10/21
Matches found: 1050

Processing file 11/21
Matches found: 1094

Processing file 12/21
Matches found: 988

Processing file 13/21
Matches found: 785

Processing file 14/21
Matches found: 784

Processing file 15/21
Matches found: 751

Processing file 16/21
Matches found: 788

Processing file 17/21
Matches found: 768

Processing file 18/21
Matches found: 631

Processing file 19/21
Matches found: 185

Processing file 20/21
Matches found: 1

Processing file 21/21
Matches found: 0

Combining matched products...
Total matched products: 30000


In [11]:
import os

processed_path = r"C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\processed"

os.makedirs(processed_path, exist_ok=True)

output_file = os.path.join(
    processed_path,
    "top_30000_product_metadata.parquet"
)

products_df.to_parquet(
    output_file,
    index=False
)

print("Saved successfully!")
print("File:", output_file)
print("Products:", len(products_df))

Saved successfully!
File: C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\processed\top_30000_product_metadata.parquet
Products: 30000


In [12]:
print(products_df.shape)

print("\nColumns:")
print(products_df.columns.tolist())

(30000, 16)

Columns:
['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author']


In [13]:
products_df.head(3)

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Amazon Home,"Walensee Large Bathroom Rug (24 x 60, Black) E...",4.3,31438,[❤ MACHINE WASHABLE: Walensee 24 Inch x 60 Inc...,[],41.99,{'hi_res': ['https://m.media-amazon.com/images...,"{'title': ['Great bath rug for bathroom, super...",Walensee,"[Home & Kitchen, Bath, Bath Rugs]","{""Color"": ""Black"", ""Pattern"": ""Solid"", ""Shape""...",B0C4CPL2RV,NaN,NaN,NaN
1,Amazon Home,CHILLOUT LIFE 20 oz Stainless Steel Tumbler wi...,4.7,18386,[5-year Warranty & 100% Satisfaction - Yes! CH...,[],13.49,{'hi_res': ['https://m.media-amazon.com/images...,{'title': ['20 oz Stainless Steel Tumbler with...,CHILLOUT LIFE,"[Home & Kitchen, Kitchen & Dining, Dining & En...","{""Brand"": ""CHILLOUT LIFE"", ""Material"": ""Stainl...",B09L626MHB,NaN,NaN,NaN
2,Amazon Home,H.VERSAILTEX Blackout Curtains 84 Inch Length ...,4.7,2696,[PAIR SET: Sold per set including 2 panels. Pa...,[],34.99,{'hi_res': ['https://m.media-amazon.com/images...,{'title': ['Bring Luxurious to Your Window wit...,H.VERSAILTEX,"[Home & Kitchen, Home Décor Products, Window T...","{""Brand"": ""H.VERSAILTEX"", ""Color"": ""Floral in ...",B0BPJ69P2H,NaN,NaN,NaN


In [14]:
print("\nMissing values:")
print(products_df.isnull().sum())


Missing values:
main_category        449
title                  0
average_rating         0
rating_number          0
features               0
description            0
price                  0
images                 0
videos                 0
store                 21
categories             0
details                0
parent_asin            0
bought_together    30000
subtitle           30000
author             30000
dtype: int64


In [15]:
print(type(products_df["images"].iloc[0]))
print(products_df["images"].iloc[0])

<class 'dict'>
{'hi_res': array(['https://m.media-amazon.com/images/I/61PYg8i0C5L._AC_SL1000_.jpg',
       'https://m.media-amazon.com/images/I/71E3+JuiwqL._AC_SL1000_.jpg',
       'https://m.media-amazon.com/images/I/713PRmee93L._AC_SL1000_.jpg',
       'https://m.media-amazon.com/images/I/71sJr4U95xL._AC_SL1000_.jpg',
       'https://m.media-amazon.com/images/I/61IHJxMP0sL._AC_SL1000_.jpg',
       'https://m.media-amazon.com/images/I/71LhzGRPL5L._AC_SL1000_.jpg',
       'https://m.media-amazon.com/images/I/71vtXTYnQqL._AC_SL1000_.jpg',
       'https://m.media-amazon.com/images/I/613-kVdme3L._AC_SL1000_.jpg'],
      dtype=object), 'large': array(['https://m.media-amazon.com/images/I/41nvPkmlR2L._AC_.jpg',
       'https://m.media-amazon.com/images/I/51HG1VwlQ+L._AC_.jpg',
       'https://m.media-amazon.com/images/I/61kBCsBMoxL._AC_.jpg',
       'https://m.media-amazon.com/images/I/51Q3licjIVL._AC_.jpg',
       'https://m.media-amazon.com/images/I/51l+AiEsC9L._AC_.jpg',
       'https://

In [16]:
print(type(products_df["images"].iloc[1]))

<class 'dict'>


In [17]:
def get_image_count(images):
    if not isinstance(images, dict):
        return 0

    hi_res = images.get("hi_res")

    if hi_res is None:
        return 0

    return len(hi_res)


products_df["image_count"] = products_df["images"].apply(get_image_count)

print("===== IMAGE STATISTICS =====")

print("Products:", len(products_df))

print(
    "Products with at least one image:",
    (products_df["image_count"] > 0).sum()
)

print(
    "Products without images:",
    (products_df["image_count"] == 0).sum()
)

print(
    "Total high-resolution images:",
    products_df["image_count"].sum()
)

print(
    "Average images per product:",
    products_df["image_count"].mean()
)

print(
    "Maximum images for one product:",
    products_df["image_count"].max()
)

===== IMAGE STATISTICS =====
Products: 30000
Products with at least one image: 30000
Products without images: 0
Total high-resolution images: 187788
Average images per product: 6.2596
Maximum images for one product: 37


In [18]:
def get_main_image(images):
    if not isinstance(images, dict):
        return None

    hi_res = images.get("hi_res")

    if hi_res is None or len(hi_res) == 0:
        return None

    return str(hi_res[0])


products_df["image_url"] = products_df["images"].apply(get_main_image)

print("Products with main image URL:",
      products_df["image_url"].notna().sum())

print("\nExample:")
print(products_df[["parent_asin", "title", "image_url"]].head(3))

Products with main image URL: 30000

Example:
  parent_asin                                              title  \
0  B0C4CPL2RV  Walensee Large Bathroom Rug (24 x 60, Black) E...   
1  B09L626MHB  CHILLOUT LIFE 20 oz Stainless Steel Tumbler wi...   
2  B0BPJ69P2H  H.VERSAILTEX Blackout Curtains 84 Inch Length ...   

                                           image_url  
0  https://m.media-amazon.com/images/I/61PYg8i0C5...  
1  https://m.media-amazon.com/images/I/71hTlPC2lk...  
2  https://m.media-amazon.com/images/I/71z4l+ypmB...  


In [19]:
image_url_df = products_df[
    ["parent_asin", "title", "image_url"]
].copy()

image_url_path = r"C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\processed\product_image_urls.parquet"

image_url_df.to_parquet(
    image_url_path,
    index=False
)

print("Saved successfully!")
print(image_url_path)

Saved successfully!
C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\processed\product_image_urls.parquet


In [20]:
product = products_df.iloc[0]

print("===== TITLE =====")
print(product["title"])

print("\n===== FEATURES =====")
print(product["features"])

print("\n===== DESCRIPTION =====")
print(product["description"])

print("\n===== CATEGORIES =====")
print(product["categories"])

print("\n===== DETAILS =====")
print(product["details"])

===== TITLE =====
Walensee Large Bathroom Rug (24 x 60, Black) Extra Soft and Absorbent Shaggy Bathroom Mat Machine Washable Microfiber Bath Mat for Bathroom, Non Slip Bath Mat, Luxury Bathroom Floor Mats

===== FEATURES =====
['❤ MACHINE WASHABLE: Walensee 24 Inch x 60 Inch luxury Bath Rug can be machine washed time and time again. Easy for clearance. We use the more expensive TP Rubber backing material (not PVC or glue) which is much stronger and durable for long lasting use.'
 '❤ PAMPER YOUR FEET: This cushioned bathmat for home is constructed with thousands of individual polyester micro fiber shags, sink your toes into the comfortable contentment of a bathtoom floor mat from threshold. Soft pile that soothes tired foot and shields toes from the cold floor.'
 '❤ ULTRA ABSORBENT: The microfiber shag bath rug is much more absorbant than cotton bath rugs. High-pile helps save your floors from dripping water while you’re stepping out of the bath, shower, spa, or getting ready by the sin

In [21]:
import json
import numpy as np
import pandas as pd


IMPORTANT_DETAILS = [
    "Color",
    "Material",
    "Size",
    "Shape",
    "Style",
    "Brand",
    "Room Type",
    "Pattern",
    "Product Dimensions",
    "Special Feature"
]


def clean_product_text(row):

    parts = []

    # 1. Title
    title = row["title"]

    if pd.notna(title):
        parts.append(f"Product: {title}")

    # 2. Features
    features = row["features"]

    if isinstance(features, (list, np.ndarray)):
        feature_text = " ".join(str(x) for x in features)

        if feature_text.strip():
            parts.append(f"Features: {feature_text}")

    # 3. Description
    description = row["description"]

    if isinstance(description, (list, np.ndarray)):
        description_text = " ".join(str(x) for x in description)

        if description_text.strip():
            parts.append(f"Description: {description_text}")

    elif isinstance(description, str) and description.strip():
        parts.append(f"Description: {description}")

    # 4. Categories
    categories = row["categories"]

    if isinstance(categories, (list, np.ndarray)):
        category_text = ", ".join(str(x) for x in categories)

        if category_text.strip():
            parts.append(f"Categories: {category_text}")

    # 5. Product details
    details = row["details"]

    # Convert JSON string → dictionary
    if isinstance(details, str):

        try:
            details = json.loads(details)
        except:
            details = {}

    if isinstance(details, dict):

        detail_parts = []

        for key in IMPORTANT_DETAILS:

            if key in details:

                value = details[key]

                if isinstance(value, dict):
                    continue

                detail_parts.append(
                    f"{key}: {value}"
                )

        if detail_parts:
            parts.append(
                "Attributes: " +
                ". ".join(detail_parts)
            )

    return ". ".join(parts)

In [22]:
test_text = clean_product_text(products_df.iloc[0])

print(test_text)

Product: Walensee Large Bathroom Rug (24 x 60, Black) Extra Soft and Absorbent Shaggy Bathroom Mat Machine Washable Microfiber Bath Mat for Bathroom, Non Slip Bath Mat, Luxury Bathroom Floor Mats. Features: ❤ MACHINE WASHABLE: Walensee 24 Inch x 60 Inch luxury Bath Rug can be machine washed time and time again. Easy for clearance. We use the more expensive TP Rubber backing material (not PVC or glue) which is much stronger and durable for long lasting use. ❤ PAMPER YOUR FEET: This cushioned bathmat for home is constructed with thousands of individual polyester micro fiber shags, sink your toes into the comfortable contentment of a bathtoom floor mat from threshold. Soft pile that soothes tired foot and shields toes from the cold floor. ❤ ULTRA ABSORBENT: The microfiber shag bath rug is much more absorbant than cotton bath rugs. High-pile helps save your floors from dripping water while you’re stepping out of the bath, shower, spa, or getting ready by the sink. Moisture is trapped in th

In [23]:
products_df["product_text"] = products_df.apply(
    clean_product_text,
    axis=1
)

print("Product text created successfully!")
print("Total products:", len(products_df))

Product text created successfully!
Total products: 30000


In [24]:
print(products_df[["parent_asin", "product_text"]].head(3))

  parent_asin                                       product_text
0  B0C4CPL2RV  Product: Walensee Large Bathroom Rug (24 x 60,...
1  B09L626MHB  Product: CHILLOUT LIFE 20 oz Stainless Steel T...
2  B0BPJ69P2H  Product: H.VERSAILTEX Blackout Curtains 84 Inc...


In [25]:
empty_text = (
    products_df["product_text"]
    .fillna("")
    .str.strip()
    .eq("")
    .sum()
)

print("Products with empty text:", empty_text)

Products with empty text: 0


In [26]:
processed_product_file = r"C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\processed\products_with_text.parquet"

products_df.to_parquet(
    processed_product_file,
    index=False
)

print("Saved successfully!")
print(processed_product_file)

Saved successfully!
C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\processed\products_with_text.parquet


In [27]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA version:", torch.version.cuda)
    print(
        "VRAM:",
        round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2),
        "GB"
    )

KeyboardInterrupt: 

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_path = r"C:\Users\amjad\FurnitureRecommendationAI_Models\all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

text_model = AutoModel.from_pretrained(
    model_path,
    local_files_only=True
)

text_model = text_model.to("cuda")
text_model.eval()

print("Model loaded successfully!")
print("Device:", next(text_model.parameters()).device)

In [ ]:
text = products_df["product_text"].iloc[0]

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=256
)

inputs = {
    key: value.to("cuda")
    for key, value in inputs.items()
}

with torch.no_grad():
    outputs = text_model(**inputs)

print("Model output shape:", outputs.last_hidden_state.shape)

In [ ]:
attention_mask = inputs["attention_mask"]

token_embeddings = outputs.last_hidden_state

mask = attention_mask.unsqueeze(-1).expand(
    token_embeddings.size()
).float()

sum_embeddings = torch.sum(
    token_embeddings * mask,
    dim=1
)

sum_mask = torch.clamp(
    mask.sum(dim=1),
    min=1e-9
)

embedding = sum_embeddings / sum_mask

embedding = torch.nn.functional.normalize(
    embedding,
    p=2,
    dim=1
)

print("Final embedding shape:", embedding.shape)

In [ ]:
import numpy as np
import torch
from tqdm.auto import tqdm


# Make sure the model is in evaluation mode
text_model.eval()

all_embeddings = []

batch_size = 32

texts = products_df["product_text"].tolist()

for start in tqdm(
    range(0, len(texts), batch_size),
    desc="Generating text embeddings"
):

    batch_texts = texts[start:start + batch_size]

    # Tokenize batch
    inputs = tokenizer(
        batch_texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    # Move to GPU
    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    # Generate embeddings
    with torch.no_grad():

        outputs = text_model(**inputs)

        token_embeddings = outputs.last_hidden_state

        attention_mask = inputs["attention_mask"]

        mask = attention_mask.unsqueeze(-1).expand(
            token_embeddings.size()
        ).float()

        sum_embeddings = torch.sum(
            token_embeddings * mask,
            dim=1
        )

        sum_mask = torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

        embeddings = sum_embeddings / sum_mask

        # Normalize
        embeddings = torch.nn.functional.normalize(
            embeddings,
            p=2,
            dim=1
        )

    # Move from GPU → CPU
    all_embeddings.append(
        embeddings.cpu().numpy()
    )

# Combine all batches
text_embeddings = np.vstack(all_embeddings)

print("Embedding generation complete!")
print("Shape:", text_embeddings.shape)

In [ ]:
embedding_path = r"C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\processed\text_embeddings.npy"

np.save(
    embedding_path,
    text_embeddings
)

print("Text embeddings saved!")
print(embedding_path)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Select one product
product_index = 0

query_embedding = text_embeddings[product_index].reshape(1, -1)

# Calculate similarity with every product
similarities = cosine_similarity(
    query_embedding,
    text_embeddings
)[0]

# Get the 11 most similar products
top_indices = np.argsort(similarities)[::-1][:11]

print("QUERY PRODUCT:")
print(products_df.iloc[product_index]["title"])

print("\n===== SIMILAR PRODUCTS =====")

for rank, idx in enumerate(top_indices[1:], start=1):

    print(
        f"\n{rank}. "
        f"{products_df.iloc[idx]['title']}"
    )

    print(
        f"Similarity: {similarities[idx]:.4f}"
    )

In [ ]:
import os

interaction_path = r"C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\raw"

for root, dirs, files in os.walk(interaction_path):
    for file in files:  
        if file.endswith((".csv", ".json", ".json.gz", ".parquet")):
            print(os.path.join(root, file))

In [ ]:
import pandas as pd

interaction_path = r"C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\raw\Home_and_Kitchen.train.csv"

sample = pd.read_csv(
    interaction_path,
    nrows=10
)

print("Columns:")
print(sample.columns.tolist())

print("\nFirst 10 rows:")
display(sample)

In [ ]:
import pandas as pd
import os

interaction_path = r"C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\raw\Home_and_Kitchen.train.csv"

# Products in our recommendation catalog
catalog_products = set(products_df["parent_asin"].astype(str))

print("Products in catalog:", len(catalog_products))

filtered_chunks = []

chunk_size = 500_000

for chunk_number, chunk in enumerate(
    pd.read_csv(
        interaction_path,
        chunksize=chunk_size
    ),
    start=1
):

    # Keep only products from our 30,000-product catalog
    filtered = chunk[
        chunk["parent_asin"].astype(str).isin(catalog_products)
    ]

    if len(filtered) > 0:
        filtered_chunks.append(filtered)

    print(
        f"Chunk {chunk_number}: "
        f"{len(chunk):,} rows → "
        f"{len(filtered):,} matching interactions"
    )

print("\nCombining filtered chunks...")

interactions_df = pd.concat(
    filtered_chunks,
    ignore_index=True
)

print("\n===== FILTERED INTERACTIONS =====")
print("Total interactions:", len(interactions_df))
print("Unique users:", interactions_df["user_id"].nunique())
print("Unique products:", interactions_df["parent_asin"].nunique())

In [ ]:
filtered_interaction_path = r"C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\processed\filtered_interactions.parquet"

interactions_df.to_parquet(
    filtered_interaction_path,
    index=False
)

print("Filtered interactions saved successfully!")
print(filtered_interaction_path)

In [ ]:
print("===== INTERACTION DATA =====")

print("Total interactions:", len(interactions_df))

print(
    "Unique users:",
    interactions_df["user_id"].nunique()
)

print(
    "Unique products:",
    interactions_df["parent_asin"].nunique()
)

print("\n===== RATINGS =====")

print(
    interactions_df["rating"]
    .value_counts()
    .sort_index()
)

print("\n===== MISSING VALUES =====")

print(
    interactions_df.isna().sum()
)

In [ ]:
user_activity = (
    interactions_df
    .groupby("user_id")
    .size()
)

print("===== USER ACTIVITY =====")

print("Minimum interactions:", user_activity.min())
print("Maximum interactions:", user_activity.max())
print("Average interactions:", user_activity.mean())
print("Median interactions:", user_activity.median())

print("\n===== USER ACTIVITY PERCENTILES =====")

print(
    user_activity.quantile(
        [0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

In [ ]:
def rating_to_preference(rating):
    if rating >= 4:
        return 1.0
    elif rating == 3:
        return 0.2
    else:
        return -1.0


interactions_df["preference"] = (
    interactions_df["rating"]
    .apply(rating_to_preference)
)

print(
    interactions_df[
        ["rating", "preference"]
    ].head(20)
)

In [ ]:
print(
    interactions_df["preference"]
    .value_counts()
    .sort_index()
)

In [ ]:
interactions_df["datetime"] = pd.to_datetime(
    interactions_df["timestamp"],
    unit="ms"
)

print(
    interactions_df[
        ["user_id", "parent_asin", "rating", "datetime"]
    ].head()
)

print("\nDate range:")
print("Start:", interactions_df["datetime"].min())
print("End:", interactions_df["datetime"].max())

In [ ]:
year_counts = (
    interactions_df["datetime"]
    .dt.year
    .value_counts()
    .sort_index()
)

print(year_counts)

In [ ]:
print("\nCumulative interaction percentage:")

cumulative = year_counts.cumsum()
percentage = cumulative / len(interactions_df) * 100

print(
    pd.DataFrame({
        "interactions": year_counts,
        "cumulative_%": percentage.round(2)
    })
)

In [ ]:
train_df = interactions_df[
    interactions_df["datetime"].dt.year <= 2020
].copy()

test_df = interactions_df[
    interactions_df["datetime"].dt.year >= 2021
].copy()

print("===== TRAIN / TEST SPLIT =====")

print("Training interactions:", len(train_df))
print("Test interactions:", len(test_df))

print(
    "Training users:",
    train_df["user_id"].nunique()
)

print(
    "Test users:",
    test_df["user_id"].nunique()
)

print(
    "Training products:",
    train_df["parent_asin"].nunique()
)

print(
    "Test products:",
    test_df["parent_asin"].nunique()
)

In [ ]:
from scipy.sparse import csr_matrix
import numpy as np

# Create user and product ID mappings from TRAINING data

user_ids = train_df["user_id"].unique()
product_ids = train_df["parent_asin"].unique()

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

product_to_index = {
    product_id: index
    for index, product_id in enumerate(product_ids)
}

print("Users:", len(user_to_index))
print("Products:", len(product_to_index))

In [ ]:
rows = train_df["user_id"].map(user_to_index).to_numpy()

cols = train_df["parent_asin"].map(product_to_index).to_numpy()

values = train_df["preference"].to_numpy(dtype=np.float32)

user_item_matrix = csr_matrix(
    (values, (rows, cols)),
    shape=(
        len(user_to_index),
        len(product_to_index)
    )
)

print("Sparse matrix created!")
print("Shape:", user_item_matrix.shape)
print("Non-zero interactions:", user_item_matrix.nnz)

In [ ]:
import implicit

print("Implicit version:", implicit.__version__)

In [ ]:
from implicit.als import AlternatingLeastSquares

als_model = AlternatingLeastSquares(
    factors=64,
    regularization=0.1,
    iterations=15,
    random_state=42
)

print("Training ALS with correct orientation...")

als_model.fit(
    user_item_matrix
)

print("ALS training complete!")

In [ ]:
print("User factors:", als_model.user_factors.shape)
print("Item factors:", als_model.item_factors.shape)

In [ ]:
# Convert product IDs to a normal NumPy array
product_ids_array = np.array(product_ids)

# Select a user who exists in the training data
test_user_id = train_df["user_id"].iloc[0]

# Convert Amazon user ID → integer index
user_index = user_to_index[test_user_id]

# Generate 10 recommendations
recommendations, scores = als_model.recommend(
    user_index,
    user_item_matrix[user_index],
    N=10,
    filter_already_liked_items=True
)

print("USER:", test_user_id)

print("\n===== PERSONALIZED RECOMMENDATIONS =====")

for rank, (product_index, score) in enumerate(
    zip(recommendations, scores),
    start=1
):

    product_id = product_ids_array[int(product_index)]

    matching_product = products_df[
        products_df["parent_asin"] == product_id
    ]

    if len(matching_product) > 0:

        title = matching_product["title"].iloc[0]

    else:

        title = "Title not found"

    print(f"\n{rank}. {title}")
    print(f"   ASIN: {product_id}")
    print(f"   ALS Score: {score:.4f}")

In [ ]:
import numpy as np
from tqdm.auto import tqdm

# Test users who also existed during training
test_users = [
    user_id
    for user_id in test_df["user_id"].unique()
    if user_id in user_to_index
]

print("Test users with training history:", len(test_users))

# Start with 1,000 users for testing
np.random.seed(42)

if len(test_users) > 1000:
    evaluation_users = np.random.choice(
        test_users,
        size=1000,
        replace=False
    )
else:
    evaluation_users = test_users

print("Users used for evaluation:", len(evaluation_users))

In [ ]:
def dcg_at_k(relevant_items, recommended_items, k=10):

    recommended_items = recommended_items[:k]

    score = 0.0

    for position, item in enumerate(recommended_items):
        if item in relevant_items:
            score += 1 / np.log2(position + 2)

    return score


def ndcg_at_k(relevant_items, recommended_items, k=10):

    if len(relevant_items) == 0:
        return 0.0

    actual_dcg = dcg_at_k(
        relevant_items,
        recommended_items,
        k
    )

    ideal_hits = min(len(relevant_items), k)

    ideal_dcg = sum(
        1 / np.log2(position + 2)
        for position in range(ideal_hits)
    )

    return actual_dcg / ideal_dcg


recall_scores = []
hit_scores = []
ndcg_scores = []

for user_id in tqdm(
    evaluation_users,
    desc="Evaluating ALS"
):

    user_index = user_to_index[user_id]

    # Actual products the user interacted with in TEST
    actual_products = set(
        test_df.loc[
            test_df["user_id"] == user_id,
            "parent_asin"
        ]
    )

    if len(actual_products) == 0:
        continue

    # Convert actual product IDs to training indices
    actual_indices = {
        product_to_index[p]
        for p in actual_products
        if p in product_to_index
    }

    if len(actual_indices) == 0:
        continue

    # Generate recommendations
    recommended_indices, scores = als_model.recommend(
        user_index,
        user_item_matrix[user_index],
        N=10,
        filter_already_liked_items=True
    )

    recommended_indices = list(
        map(int, recommended_indices)
    )

    # Number of relevant products found
    hits = len(
        set(recommended_indices) &
        actual_indices
    )

    recall = hits / len(actual_indices)

    hit_rate = 1.0 if hits > 0 else 0.0

    ndcg = ndcg_at_k(
        actual_indices,
        recommended_indices,
        k=10
    )

    recall_scores.append(recall)
    hit_scores.append(hit_rate)
    ndcg_scores.append(ndcg)


print("\n===== ALS EVALUATION =====")

print(
    "Recall@10:",
    round(np.mean(recall_scores), 4)
)

print(
    "Hit Rate@10:",
    round(np.mean(hit_scores), 4)
)

print(
    "NDCG@10:",
    round(np.mean(ndcg_scores), 4)
)

In [ ]:
confidence_train = train_df.copy()

confidence_train["confidence"] = (
    confidence_train["rating"]
    .replace({
        5:5,
        4:4,
        3:2,
        2:1,
        1:1
    })
)

alpha = 40

confidence_train["confidence"] *= alpha

In [ ]:
rows = confidence_train["user_id"].map(user_to_index).to_numpy()

cols = confidence_train["parent_asin"].map(product_to_index).to_numpy()

values = confidence_train["confidence"].to_numpy(dtype=np.float32)

confidence_matrix = csr_matrix(
    (values,(rows,cols)),
    shape=user_item_matrix.shape
)

print(confidence_matrix.shape)

In [ ]:
from implicit.als import AlternatingLeastSquares

als_model_v2 = AlternatingLeastSquares(
    factors=128,
    regularization=0.05,
    iterations=20,
    random_state=42
)

print("Training ALS v2...")

als_model_v2.fit(confidence_matrix)

print("Training complete!")

In [ ]:
recall_scores_v2 = []
hit_scores_v2 = []
ndcg_scores_v2 = []

for user_id in tqdm(
    evaluation_users,
    desc="Evaluating ALS v2"
):

    user_index = user_to_index[user_id]

    actual_products = set(
        test_df.loc[
            test_df["user_id"] == user_id,
            "parent_asin"
        ]
    )

    if len(actual_products) == 0:
        continue

    actual_indices = {
        product_to_index[p]
        for p in actual_products
        if p in product_to_index
    }

    if len(actual_indices) == 0:
        continue

    recommended_indices, scores = als_model_v2.recommend(
        user_index,
        user_item_matrix[user_index],
        N=10,
        filter_already_liked_items=True
    )

    recommended_indices = list(
        map(int, recommended_indices)
    )

    hits = len(
        set(recommended_indices) &
        actual_indices
    )

    recall = hits / len(actual_indices)

    hit_rate = 1.0 if hits > 0 else 0.0

    ndcg = ndcg_at_k(
        actual_indices,
        recommended_indices,
        k=10
    )

    recall_scores_v2.append(recall)
    hit_scores_v2.append(hit_rate)
    ndcg_scores_v2.append(ndcg)


print("\n===== ALS V2 EVALUATION =====")

print(
    "Recall@10:",
    round(np.mean(recall_scores_v2), 4)
)

print(
    "Hit Rate@10:",
    round(np.mean(hit_scores_v2), 4)
)

print(
    "NDCG@10:",
    round(np.mean(ndcg_scores_v2), 4)
)

In [ ]:
test_user_id = evaluation_users[0]

print("Test user:", test_user_id)

In [ ]:
user_history = train_df[
    train_df["user_id"] == test_user_id
]

print("User interactions:", len(user_history))

print("\nUser's previous products:")

for asin in user_history["parent_asin"].head(10):
    match = products_df[
        products_df["parent_asin"] == asin
    ]

    if len(match) > 0:
        print("-", match["title"].iloc[0])

In [ ]:
# Find users with at least 2 positive training interactions

positive_user_counts = (
    train_df[train_df["rating"] >= 4]
    .groupby("user_id")
    .size()
)

eligible_users = positive_user_counts[
    positive_user_counts >= 2
].index

print("Users with at least 2 positive interactions:", len(eligible_users))

# Select one user
test_user_id = eligible_users[0]

print("\nSelected user:", test_user_id)

In [ ]:
user_history = train_df[
    train_df["user_id"] == test_user_id
]

positive_history = user_history[
    user_history["rating"] >= 4
]

print("Total interactions:", len(user_history))
print("Positive interactions:", len(positive_history))

print("\nPositive products:")

for asin in positive_history["parent_asin"]:
    
    match = products_df[
        products_df["parent_asin"] == asin
    ]
    
    if len(match) > 0:
        print("-", match["title"].iloc[0])

In [ ]:
positive_history = user_history[
    user_history["rating"] >= 4
]

history_indices = [
    product_to_index[asin]
    for asin in positive_history["parent_asin"]
    if asin in product_to_index
]

print("Positive products:", len(history_indices))

In [ ]:
product_df_index = {
    asin: idx
    for idx, asin in enumerate(
        products_df["parent_asin"]
    )
}

In [ ]:
text_history_indices = [
    product_df_index[asin]
    for asin in positive_history["parent_asin"]
    if asin in product_df_index
]

user_text_embedding = text_embeddings[
    text_history_indices
].mean(axis=0)

user_text_embedding = (
    user_text_embedding /
    np.linalg.norm(user_text_embedding)
)

print(
    "User text embedding shape:",
    user_text_embedding.shape
)

In [ ]:
als_recommendations, als_scores = als_model_v2.recommend(
    user_to_index[test_user_id],
    user_item_matrix[user_to_index[test_user_id]],
    N=100,
    filter_already_liked_items=True
)

print("ALS candidates:", len(als_recommendations))

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Convert ALS product indices → ASINs
candidate_asins = [
    product_ids_array[int(idx)]
    for idx in als_recommendations
]

# Convert ASINs → rows in products_df/text_embeddings
candidate_text_indices = [
    product_df_index[asin]
    for asin in candidate_asins
]

# Get MiniLM embeddings for the 100 candidates
candidate_embeddings = text_embeddings[
    candidate_text_indices
]

# Compare user's text profile with each candidate
text_scores = cosine_similarity(
    user_text_embedding.reshape(1, -1),
    candidate_embeddings
)[0]

print("Text scores calculated:", len(text_scores))
print("Example scores:", text_scores[:5])

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# Convert scores to arrays
als_scores_array = np.array(als_scores).reshape(-1, 1)
text_scores_array = np.array(text_scores).reshape(-1, 1)

# Normalize ALS scores
als_normalized = MinMaxScaler().fit_transform(
    als_scores_array
).flatten()

# Normalize text similarity scores
text_normalized = MinMaxScaler().fit_transform(
    text_scores_array
).flatten()

# Combine both signals
hybrid_scores = (
    0.60 * als_normalized
    +
    0.40 * text_normalized
)

print("Hybrid scores calculated!")
print("Number of hybrid scores:", len(hybrid_scores))

In [ ]:
ranking = np.argsort(hybrid_scores)[::-1][:10]

print("\n===== HYBRID TOP 10 RECOMMENDATIONS =====")

for rank, position in enumerate(ranking, start=1):

    asin = candidate_asins[position]

    match = products_df[
        products_df["parent_asin"] == asin
    ]

    if len(match) == 0:
        continue

    title = match["title"].iloc[0]

    print(f"\n{rank}. {title}")
    print(f"   ASIN: {asin}")
    print(f"   ALS Score: {als_normalized[position]:.4f}")
    print(f"   Text Score: {text_normalized[position]:.4f}")
    print(f"   Hybrid Score: {hybrid_scores[position]:.4f}")

In [ ]:
hybrid_recall_scores = []
hybrid_hit_scores = []
hybrid_ndcg_scores = []

for user_id in tqdm(
    evaluation_users,
    desc="Evaluating Hybrid"
):

    # Skip users without a trained history
    if user_id not in user_to_index:
        continue

    user_index = user_to_index[user_id]

    # Actual products from the TEST period
    actual_products = set(
        test_df.loc[
            test_df["user_id"] == user_id,
            "parent_asin"
        ]
    )

    if len(actual_products) == 0:
        continue

    # Products that exist in our training catalog
    actual_indices = {
        product_to_index[p]
        for p in actual_products
        if p in product_to_index
    }

    if len(actual_indices) == 0:
        continue

    # ------------------------------------------------
    # 1. Get 100 ALS candidates
    # ------------------------------------------------

    als_recommendations, als_scores = als_model_v2.recommend(
        user_index,
        user_item_matrix[user_index],
        N=100,
        filter_already_liked_items=True
    )

    candidate_asins = [
        product_ids_array[int(idx)]
        for idx in als_recommendations
    ]

    # ------------------------------------------------
    # 2. Create user's text profile
    # ------------------------------------------------

    user_history = train_df[
        train_df["user_id"] == user_id
    ]

    positive_history = user_history[
        user_history["rating"] >= 4
    ]

    text_history_indices = [
        product_df_index[asin]
        for asin in positive_history["parent_asin"]
        if asin in product_df_index
    ]

    # If user has no positive history, skip
    if len(text_history_indices) == 0:
        continue

    user_text_embedding = text_embeddings[
        text_history_indices
    ].mean(axis=0)

    norm = np.linalg.norm(user_text_embedding)

    if norm == 0:
        continue

    user_text_embedding = (
        user_text_embedding / norm
    )

    # ------------------------------------------------
    # 3. Calculate text similarity
    # ------------------------------------------------

    candidate_text_indices = [
        product_df_index[asin]
        for asin in candidate_asins
    ]

    candidate_embeddings = text_embeddings[
        candidate_text_indices
    ]

    text_scores = cosine_similarity(
        user_text_embedding.reshape(1, -1),
        candidate_embeddings
    )[0]

    # ------------------------------------------------
    # 4. Normalize both scores
    # ------------------------------------------------

    als_normalized = MinMaxScaler().fit_transform(
        np.array(als_scores).reshape(-1, 1)
    ).flatten()

    text_normalized = MinMaxScaler().fit_transform(
        np.array(text_scores).reshape(-1, 1)
    ).flatten()

    # ------------------------------------------------
    # 5. Hybrid score
    # ------------------------------------------------

    hybrid_scores = (
        0.60 * als_normalized
        +
        0.40 * text_normalized
    )

    # ------------------------------------------------
    # 6. Select final Top 10
    # ------------------------------------------------

    ranking = np.argsort(
        hybrid_scores
    )[::-1][:10]

    recommended_asins = [
        candidate_asins[position]
        for position in ranking
    ]

    # Convert recommended ASINs to product indices
    recommended_indices = [
        product_to_index[asin]
        for asin in recommended_asins
        if asin in product_to_index
    ]

    # ------------------------------------------------
    # 7. Calculate metrics
    # ------------------------------------------------

    hits = len(
        set(recommended_indices)
        &
        actual_indices
    )

    recall = hits / len(actual_indices)

    hit_rate = 1.0 if hits > 0 else 0.0

    ndcg = ndcg_at_k(
        actual_indices,
        recommended_indices,
        k=10
    )

    hybrid_recall_scores.append(recall)
    hybrid_hit_scores.append(hit_rate)
    hybrid_ndcg_scores.append(ndcg)


print("\n===== HYBRID EVALUATION =====")

print(
    "Recall@10:",
    round(np.mean(hybrid_recall_scores), 4)
)

print(
    "Hit Rate@10:",
    round(np.mean(hybrid_hit_scores), 4)
)

print(
    "NDCG@10:",
    round(np.mean(hybrid_ndcg_scores), 4)
)

print(
    "\nUsers successfully evaluated:",
    len(hybrid_recall_scores)
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm
import numpy as np

weights = [
    (0.9, 0.1),
    (0.8, 0.2),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),
    (0.4, 0.6),
    (0.3, 0.7),
    (0.2, 0.8),
    (0.1, 0.9)
]

results = []

# Pre-calculate useful mappings
product_df_index = {
    asin: idx
    for idx, asin in enumerate(products_df["parent_asin"])
}

for als_weight, text_weight in weights:

    recall_scores = []
    hit_scores = []
    ndcg_scores = []

    for user_id in evaluation_users:

        if user_id not in user_to_index:
            continue

        user_index = user_to_index[user_id]

        # Future products from TEST
        actual_products = set(
            test_df.loc[
                test_df["user_id"] == user_id,
                "parent_asin"
            ]
        )

        if not actual_products:
            continue

        actual_indices = {
            product_to_index[p]
            for p in actual_products
            if p in product_to_index
        }

        if not actual_indices:
            continue

        # -----------------------------
        # ALS candidates
        # -----------------------------

        als_recommendations, als_scores = als_model_v2.recommend(
            user_index,
            user_item_matrix[user_index],
            N=100,
            filter_already_liked_items=True
        )

        candidate_asins = [
            product_ids_array[int(idx)]
            for idx in als_recommendations
        ]

        # -----------------------------
        # User text profile
        # -----------------------------

        user_history = train_df[
            train_df["user_id"] == user_id
        ]

        positive_history = user_history[
            user_history["rating"] >= 4
        ]

        text_history_indices = [
            product_df_index[asin]
            for asin in positive_history["parent_asin"]
            if asin in product_df_index
        ]

        if not text_history_indices:
            continue

        user_text_embedding = text_embeddings[
            text_history_indices
        ].mean(axis=0)

        norm = np.linalg.norm(user_text_embedding)

        if norm == 0:
            continue

        user_text_embedding /= norm

        # -----------------------------
        # Text similarity
        # -----------------------------

        candidate_text_indices = [
            product_df_index[asin]
            for asin in candidate_asins
        ]

        candidate_embeddings = text_embeddings[
            candidate_text_indices
        ]

        text_scores = cosine_similarity(
            user_text_embedding.reshape(1, -1),
            candidate_embeddings
        )[0]

        # -----------------------------
        # Normalize scores
        # -----------------------------

        als_normalized = MinMaxScaler().fit_transform(
            np.array(als_scores).reshape(-1, 1)
        ).flatten()

        text_normalized = MinMaxScaler().fit_transform(
            np.array(text_scores).reshape(-1, 1)
        ).flatten()

        # -----------------------------
        # Hybrid score
        # -----------------------------

        hybrid_scores = (
            als_weight * als_normalized
            +
            text_weight * text_normalized
        )

        # -----------------------------
        # Top 10
        # -----------------------------

        ranking = np.argsort(
            hybrid_scores
        )[::-1][:10]

        recommended_indices = [
            product_to_index[candidate_asins[position]]
            for position in ranking
            if candidate_asins[position] in product_to_index
        ]

        # -----------------------------
        # Metrics
        # -----------------------------

        hits = len(
            set(recommended_indices)
            &
            actual_indices
        )

        recall = hits / len(actual_indices)

        hit_rate = 1.0 if hits > 0 else 0.0

        ndcg = ndcg_at_k(
            actual_indices,
            recommended_indices,
            k=10
        )

        recall_scores.append(recall)
        hit_scores.append(hit_rate)
        ndcg_scores.append(ndcg)

    results.append({
        "ALS Weight": als_weight,
        "Text Weight": text_weight,
        "Recall@10": np.mean(recall_scores),
        "Hit Rate@10": np.mean(hit_scores),
        "NDCG@10": np.mean(ndcg_scores),
        "Users": len(recall_scores)
    })

    print(
        f"ALS {als_weight:.1f} / Text {text_weight:.1f} → "
        f"Recall: {np.mean(recall_scores):.4f}, "
        f"Hit Rate: {np.mean(hit_scores):.4f}, "
        f"NDCG: {np.mean(ndcg_scores):.4f}, "
        f"Users: {len(recall_scores)}"
    )

In [ ]:
import requests
from PIL import Image
from io import BytesIO

test_urls = products_df["image_url"].head(10).tolist()

success = 0

for i, url in enumerate(test_urls, 1):
    try:
        response = requests.get(url, timeout=10)

        if response.status_code == 200:
            image = Image.open(BytesIO(response.content))
            print(f"{i}. SUCCESS - {image.size}")
            success += 1
        else:
            print(f"{i}. FAILED - HTTP {response.status_code}")

    except Exception as e:
        print(f"{i}. FAILED - {e}")

print(f"\nWorking images: {success}/10")

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch
import torchvision

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch
from torchvision.models import resnet50, ResNet50_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained ResNet-50
weights = ResNet50_Weights.DEFAULT
resnet = resnet50(weights=weights)

# Remove the final classification layer
resnet.fc = torch.nn.Identity()

# Move model to GPU
resnet = resnet.to(device)

# Evaluation mode
resnet.eval()

print("Model loaded successfully!")
print("Device:", device)
print("Output dimension:", 2048)

In [ ]:
import requests
from PIL import Image
from io import BytesIO

# Get first product image
image_url = products_df["image_url"].iloc[0]

# Download image
response = requests.get(image_url, timeout=20)
image = Image.open(BytesIO(response.content)).convert("RGB")

print("Image size:", image.size)

In [ ]:
# ResNet-50 preprocessing
preprocess = weights.transforms()

image_tensor = preprocess(image)
image_tensor = image_tensor.unsqueeze(0).to(device)

# Generate image embedding
with torch.no_grad():
    image_embedding = resnet(image_tensor)

print("Image embedding shape:", image_embedding.shape)

In [ ]:
from pathlib import Path
import numpy as np

# External SSD
IMAGE_EMBEDDING_DIR = Path(
    r"E:\FurnitureRecommendationAI_ImageEmbeddings"
)

IMAGE_EMBEDDING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

embedding_path = IMAGE_EMBEDDING_DIR / "image_embeddings.dat"
status_path = IMAGE_EMBEDDING_DIR / "image_status.npy"

print("Storage folder:")
print(IMAGE_EMBEDDING_DIR)

print("\nEmbedding file:")
print(embedding_path)

In [ ]:
N_PRODUCTS = len(products_df)
EMBEDDING_DIM = 2048

print("Products:", N_PRODUCTS)
print("Embedding dimension:", EMBEDDING_DIM)

embedding_size_gb = (
    N_PRODUCTS * EMBEDDING_DIM * 4
) / (1024 ** 3)

print(
    f"Required embedding storage: "
    f"{embedding_size_gb:.2f} GB"
)

In [ ]:
if not embedding_path.exists():

    image_embeddings = np.memmap(
        embedding_path,
        dtype="float32",
        mode="w+",
        shape=(N_PRODUCTS, EMBEDDING_DIM)
    )

    # Initialize with zeros
    image_embeddings[:] = 0
    image_embeddings.flush()

    # 0 = not processed
    # 1 = successfully processed
    # -1 = failed
    image_status = np.zeros(
        N_PRODUCTS,
        dtype=np.int8
    )

    np.save(
        status_path,
        image_status
    )

    print("New image embedding storage created!")

else:

    image_embeddings = np.memmap(
        embedding_path,
        dtype="float32",
        mode="r+",
        shape=(N_PRODUCTS, EMBEDDING_DIM)
    )

    image_status = np.load(
        status_path
    )

    print("Existing image embedding storage loaded!")

print(
    "Embedding shape:",
    image_embeddings.shape
)

print(
    "Already processed:",
    np.sum(image_status == 1)
)

print(
    "Failed:",
    np.sum(image_status == -1)
)

print(
    "Remaining:",
    np.sum(image_status == 0)
)

In [ ]:
import requests
import numpy as np
import torch
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm

# ---------------------------------------
# Settings
# ---------------------------------------

TEST_COUNT = 100
BATCH_SIZE = 32

print("Device:", device)
print("Testing first:", TEST_COUNT, "images")

# ---------------------------------------
# Download + preprocess images
# ---------------------------------------

def download_image(url, timeout=20):
    try:
        response = requests.get(
            url,
            timeout=timeout,
            headers={
                "User-Agent": "Mozilla/5.0"
            }
        )

        if response.status_code != 200:
            return None

        image = Image.open(
            BytesIO(response.content)
        ).convert("RGB")

        return image

    except Exception:
        return None


# ---------------------------------------
# Process images in batches
# ---------------------------------------

successful = 0
failed = 0

for start in tqdm(
    range(0, TEST_COUNT, BATCH_SIZE),
    desc="Generating image embeddings"
):

    end = min(
        start + BATCH_SIZE,
        TEST_COUNT
    )

    batch_tensors = []
    batch_indices = []

    # Download images
    for idx in range(start, end):

        url = products_df["image_url"].iloc[idx]

        image = download_image(url)

        if image is None:
            image_status[idx] = -1
            failed += 1
            continue

        try:
            tensor = preprocess(image)
            batch_tensors.append(tensor)
            batch_indices.append(idx)

        except Exception:
            image_status[idx] = -1
            failed += 1

    # Nothing to process
    if len(batch_tensors) == 0:
        continue

    # Create GPU batch
    batch = torch.stack(
        batch_tensors
    ).to(device)

    # Generate embeddings
    with torch.no_grad():
        embeddings = resnet(batch)

    # Move GPU → CPU
    embeddings = embeddings.cpu().numpy()

    # Save embeddings
    for position, idx in enumerate(batch_indices):

        image_embeddings[idx] = embeddings[position]

        image_status[idx] = 1

        successful += 1

    # Save progress
    image_embeddings.flush()

    np.save(
        status_path,
        image_status
    )

print("\n===== IMAGE TEST COMPLETE =====")

print("Successful:", successful)
print("Failed:", failed)
print(
    "Remaining:",
    np.sum(image_status == 0)
)

In [ ]:
'''import requests
import numpy as np
import torch
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm
import time

# ==========================================
# SETTINGS
# ==========================================

BATCH_SIZE = 32
MAX_RETRIES = 3
TIMEOUT = 20

print("Device:", device)
print("Total products:", len(products_df))

# ==========================================
# IMAGE DOWNLOAD FUNCTION
# ==========================================

def download_image(url, timeout=20, retries=3):

    for attempt in range(retries):

        try:
            response = requests.get(
                url,
                timeout=timeout,
                headers={
                    "User-Agent": "Mozilla/5.0"
                }
            )

            if response.status_code != 200:
                continue

            image = Image.open(
                BytesIO(response.content)
            ).convert("RGB")

            return image

        except Exception:
            if attempt < retries - 1:
                time.sleep(1)

    return None


# ==========================================
# FIND PRODUCTS THAT STILL NEED PROCESSING
# ==========================================

remaining_indices = np.where(
    image_status != 1
)[0]

print(
    "Images still needing processing:",
    len(remaining_indices)
)

print(
    "Already successful:",
    np.sum(image_status == 1)
)

print(
    "Previously failed:",
    np.sum(image_status == -1)
)


# ==========================================
# PROCESS IMAGES
# ==========================================

successful_this_run = 0
failed_this_run = 0

for batch_start in tqdm(
    range(
        0,
        len(remaining_indices),
        BATCH_SIZE
    ),
    desc="Generating image embeddings"
):

    batch_indices_to_process = remaining_indices[
        batch_start:
        batch_start + BATCH_SIZE
    ]

    tensors = []
    valid_indices = []

    # --------------------------------------
    # Download + preprocess
    # --------------------------------------

    for idx in batch_indices_to_process:

        url = products_df[
            "image_url"
        ].iloc[idx]

        image = download_image(
            url,
            timeout=TIMEOUT,
            retries=MAX_RETRIES
        )

        if image is None:

            image_status[idx] = -1
            failed_this_run += 1

            continue

        try:

            tensor = preprocess(image)

            tensors.append(tensor)
            valid_indices.append(idx)

        except Exception:

            image_status[idx] = -1
            failed_this_run += 1

    # --------------------------------------
    # GPU processing
    # --------------------------------------

    if len(tensors) > 0:

        batch_tensor = torch.stack(
            tensors
        ).to(device)

        with torch.no_grad():

            batch_embeddings = resnet(
                batch_tensor
            )

        batch_embeddings = (
            batch_embeddings
            .cpu()
            .numpy()
        )

        # ----------------------------------
        # Save embeddings
        # ----------------------------------

        for position, idx in enumerate(
            valid_indices
        ):

            image_embeddings[idx] = (
                batch_embeddings[position]
            )

            image_status[idx] = 1

            successful_this_run += 1

    # --------------------------------------
    # Save progress after EVERY batch
    # --------------------------------------

    image_embeddings.flush()

    np.save(
        status_path,
        image_status
    )


# ==========================================
# FINAL STATUS
# ==========================================

print("\n================================")
print("IMAGE EMBEDDING GENERATION DONE")
print("================================")

print(
    "Successful:",
    np.sum(image_status == 1)
)

print(
    "Failed:",
    np.sum(image_status == -1)
)

print(
    "Remaining:",
    np.sum(image_status == 0)
)

print(
    "Successful this run:",
    successful_this_run
)

print(
    "Failed this run:",
    failed_this_run
)'''

In [1]:
'''import requests
import numpy as np
import torch
from PIL import Image
from io import BytesIO
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor, as_completed
import time

# ==========================================
# SETTINGS
# ==========================================

MAX_WORKERS = 6
BATCH_SIZE = 32
MAX_RETRIES = 3
TIMEOUT = (10, 30)

print("Device:", device)

# ==========================================
# GET FAILED INDICES ONLY
# ==========================================

failed_indices = np.where(
    image_status == -1
)[0]

print("Failed images to retry:", len(failed_indices))
print("Already successful:", np.sum(image_status == 1))


# ==========================================
# DOWNLOAD FUNCTION
# ==========================================

def download_image_for_retry(idx):

    url = products_df["image_url"].iloc[idx]

    for attempt in range(MAX_RETRIES):

        try:
            response = requests.get(
                url,
                timeout=TIMEOUT,
                headers={
                    "User-Agent": (
                        "Mozilla/5.0 "
                        "(Windows NT 10.0; Win64; x64) "
                        "AppleWebKit/537.36 "
                        "Chrome/151.0 Safari/537.36"
                    )
                }
            )

            if response.status_code == 200:

                image = Image.open(
                    BytesIO(response.content)
                ).convert("RGB")

                return idx, image

        except Exception:
            pass

        # Small delay before retry
        time.sleep(1)

    return idx, None


# ==========================================
# RETRY IN BATCHES
# ==========================================

successful_retry = 0
failed_retry = 0

for batch_start in tqdm(
    range(
        0,
        len(failed_indices),
        BATCH_SIZE
    ),
    desc="Retrying failed images"
):

    batch_indices = failed_indices[
        batch_start:
        batch_start + BATCH_SIZE
    ]

    downloaded_images = []

    # --------------------------------------
    # Download concurrently
    # --------------------------------------

    with ThreadPoolExecutor(
        max_workers=MAX_WORKERS
    ) as executor:

        futures = [
            executor.submit(
                download_image_for_retry,
                idx
            )
            for idx in batch_indices
        ]

        for future in as_completed(futures):

            idx, image = future.result()

            if image is not None:

                try:
                    tensor = preprocess(image)

                    downloaded_images.append(
                        (idx, tensor)
                    )

                except Exception:

                    image_status[idx] = -1

            else:

                image_status[idx] = -1

    # --------------------------------------
    # GPU processing
    # --------------------------------------

    if downloaded_images:

        tensors = torch.stack([
            item[1]
            for item in downloaded_images
        ]).to(device)

        indices = [
            item[0]
            for item in downloaded_images
        ]

        with torch.no_grad():

            embeddings = resnet(
                tensors
            )

        embeddings = (
            embeddings
            .cpu()
            .numpy()
        )

        # ----------------------------------
        # Save successful embeddings
        # ----------------------------------

        for position, idx in enumerate(indices):

            image_embeddings[idx] = (
                embeddings[position]
            )

            image_status[idx] = 1

            successful_retry += 1

    # --------------------------------------
    # Save progress after every batch
    # --------------------------------------

    image_embeddings.flush()

    np.save(
        status_path,
        image_status
    )


# ==========================================
# FINAL RESULT
# ==========================================

print("\n======================================")
print("FAILED IMAGE RETRY COMPLETE")
print("======================================")

print(
    "Total successful:",
    np.sum(image_status == 1)
)

print(
    "Total failed:",
    np.sum(image_status == -1)
)

print(
    "Total remaining:",
    np.sum(image_status == 0)
)

print(
    "Recovered this run:",
    successful_retry
)'''

'import requests\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom io import BytesIO\nfrom tqdm.auto import tqdm\nfrom concurrent.futures import ThreadPoolExecutor, as_completed\nimport time\n\n# ==========================================\n# SETTINGS\n# ==========================================\n\nMAX_WORKERS = 6\nBATCH_SIZE = 32\nMAX_RETRIES = 3\nTIMEOUT = (10, 30)\n\nprint("Device:", device)\n\n# ==========================================\n# GET FAILED INDICES ONLY\n# ==========================================\n\nfailed_indices = np.where(\n    image_status == -1\n)[0]\n\nprint("Failed images to retry:", len(failed_indices))\nprint("Already successful:", np.sum(image_status == 1))\n\n\n# ==========================================\n# DOWNLOAD FUNCTION\n# ==========================================\n\ndef download_image_for_retry(idx):\n\n    url = products_df["image_url"].iloc[idx]\n\n    for attempt in range(MAX_RETRIES):\n\n        try:\n            response = requests.

In [2]:
'''print("===== IMAGE EMBEDDING STATUS =====")

print("Shape:", image_embeddings.shape)

print(
    "Successful:",
    np.sum(image_status == 1)
)

print(
    "Failed:",
    np.sum(image_status == -1)
)

print(
    "Remaining:",
    np.sum(image_status == 0)
)

# Check a successful embedding
successful_indices = np.where(
    image_status == 1
)[0]

sample_idx = successful_indices[0]

sample_embedding = image_embeddings[
    sample_idx
]

print("\nSample product index:", sample_idx)
print(
    "Sample embedding shape:",
    sample_embedding.shape
)

print(
    "First 10 values:",
    sample_embedding[:10]
)

print(
    "Embedding norm:",
    np.linalg.norm(sample_embedding)
)'''

'print("===== IMAGE EMBEDDING STATUS =====")\n\nprint("Shape:", image_embeddings.shape)\n\nprint(\n    "Successful:",\n    np.sum(image_status == 1)\n)\n\nprint(\n    "Failed:",\n    np.sum(image_status == -1)\n)\n\nprint(\n    "Remaining:",\n    np.sum(image_status == 0)\n)\n\n# Check a successful embedding\nsuccessful_indices = np.where(\n    image_status == 1\n)[0]\n\nsample_idx = successful_indices[0]\n\nsample_embedding = image_embeddings[\n    sample_idx\n]\n\nprint("\nSample product index:", sample_idx)\nprint(\n    "Sample embedding shape:",\n    sample_embedding.shape\n)\n\nprint(\n    "First 10 values:",\n    sample_embedding[:10]\n)\n\nprint(\n    "Embedding norm:",\n    np.linalg.norm(sample_embedding)\n)'

In [3]:
from pathlib import Path
import numpy as np

# ---------------------------------------
# Image embedding storage
# ---------------------------------------

IMAGE_EMBEDDING_DIR = Path(
    r"E:\FurnitureRecommendationAI_ImageEmbeddings"
)

embedding_path = (
    IMAGE_EMBEDDING_DIR /
    "image_embeddings.dat"
)

status_path = (
    IMAGE_EMBEDDING_DIR /
    "image_status.npy"
)

N_PRODUCTS = 30000
EMBEDDING_DIM = 2048

# Load saved embeddings
image_embeddings = np.memmap(
    embedding_path,
    dtype="float32",
    mode="r",
    shape=(N_PRODUCTS, EMBEDDING_DIM)
)

# Load processing status
image_status = np.load(status_path)

print("===== IMAGE EMBEDDINGS LOADED =====")

print("Shape:", image_embeddings.shape)

print(
    "Successful:",
    np.sum(image_status == 1)
)

print(
    "Failed:",
    np.sum(image_status == -1)
)

print(
    "Remaining:",
    np.sum(image_status == 0)
)

===== IMAGE EMBEDDINGS LOADED =====
Shape: (30000, 2048)
Successful: 28203
Failed: 1797
Remaining: 0


In [4]:
# Find products with successful image embeddings
successful_indices = np.where(
    image_status == 1
)[0]

print(
    "Products with image embeddings:",
    len(successful_indices)
)

# Check one embedding
sample_idx = successful_indices[0]

sample_embedding = image_embeddings[
    sample_idx
]

print(
    "Sample product index:",
    sample_idx
)

print(
    "Embedding shape:",
    sample_embedding.shape
)

print(
    "Embedding norm:",
    np.linalg.norm(sample_embedding)
)

Products with image embeddings: 28203
Sample product index: 0
Embedding shape: (2048,)
Embedding norm: 12.051238


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

# ==========================================
# PATHS
# ==========================================

PROCESSED_DIR = Path(
    r"C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data\amazon_2023\processed"
)

# Product image information
image_url_path = (
    PROCESSED_DIR / "product_image_urls.parquet"
)

# Product text information
product_text_path = (
    PROCESSED_DIR / "products_with_text.parquet"
)

# Text embeddings
text_embedding_path = (
    PROCESSED_DIR / "text_embeddings.npy"
)


# ==========================================
# LOAD PRODUCT DATA
# ==========================================

products_df = pd.read_parquet(
    image_url_path
)

products_text_df = pd.read_parquet(
    product_text_path
)


# ==========================================
# LOAD TEXT EMBEDDINGS
# ==========================================

text_embeddings = np.load(
    text_embedding_path
)


# ==========================================
# CHECK
# ==========================================

print("===== DATA RELOADED =====")

print(
    "Products dataframe:",
    products_df.shape
)

print(
    "Products text dataframe:",
    products_text_df.shape
)

print(
    "Text embeddings:",
    text_embeddings.shape
)

print(
    "Image embeddings:",
    image_embeddings.shape
)

===== DATA RELOADED =====
Products dataframe: (30000, 3)
Products text dataframe: (30000, 19)
Text embeddings: (30000, 384)
Image embeddings: (30000, 2048)


In [6]:
# ==========================================
# CHECK PRODUCT ORDER
# ==========================================

image_asins = (
    products_df["parent_asin"]
    .astype(str)
    .to_numpy()
)

text_asins = (
    products_text_df["parent_asin"]
    .astype(str)
    .to_numpy()
)

print("===== PRODUCT ORDER CHECK =====")

print("Image-product rows:", len(image_asins))
print("Text-product rows:", len(text_asins))

same_order = np.array_equal(
    image_asins,
    text_asins
)

print(
    "Same product order:",
    same_order
)

if same_order:
    print(
        "\n✅ Text and image embeddings use the same product order!"
    )
else:
    print(
        "\n⚠️ Product orders are different."
    )
    print(
        "STOP — do not continue until we fix the alignment."
    )

===== PRODUCT ORDER CHECK =====
Image-product rows: 30000
Text-product rows: 30000
Same product order: True

✅ Text and image embeddings use the same product order!


In [7]:
print("===== CHECKING PREVIOUS MODEL VARIABLES =====")

variables_to_check = [
    "train_df",
    "test_df",
    "als_model_v2",
    "user_to_index",
    "product_to_index",
    "product_ids_array",
    "evaluation_users"
]

for name in variables_to_check:
    print(
        f"{name}:",
        "AVAILABLE" if name in globals() else "NOT AVAILABLE"
    )

===== CHECKING PREVIOUS MODEL VARIABLES =====
train_df: NOT AVAILABLE
test_df: NOT AVAILABLE
als_model_v2: NOT AVAILABLE
user_to_index: NOT AVAILABLE
product_to_index: NOT AVAILABLE
product_ids_array: NOT AVAILABLE
evaluation_users: NOT AVAILABLE


In [43]:
# ==========================================
# RECREATE DATETIME FROM TIMESTAMP
# ==========================================

print("Available columns:")
print(interactions_df.columns.tolist())

# Amazon timestamps are in milliseconds
interactions_df["datetime"] = pd.to_datetime(
    interactions_df["timestamp"],
    unit="ms"
)

print("\n===== INTERACTIONS RELOADED =====")

print(
    "Total interactions:",
    len(interactions_df)
)

print(
    "Columns:",
    interactions_df.columns.tolist()
)

print("\nDate range:")

print(
    "Start:",
    interactions_df["datetime"].min()
)

print(
    "End:",
    interactions_df["datetime"].max()
)

Available columns:


NameError: name 'interactions_df' is not defined

In [45]:
# ==========================================
# CREATE PREFERENCE SCORE
# ==========================================

def rating_to_preference(rating):

    if rating >= 4:
        return 1.0

    elif rating == 3:
        return 0.2

    else:
        return -1.0


interactions_df["preference"] = (
    interactions_df["rating"]
    .apply(rating_to_preference)
)

print("===== PREFERENCE DISTRIBUTION =====")

print(
    interactions_df["preference"].value_counts()
)

NameError: name 'interactions_df' is not defined

In [46]:
# ==========================================
# TEMPORAL TRAIN / TEST SPLIT
# ==========================================

interactions_df["datetime"] = pd.to_datetime(
    interactions_df["datetime"]
)

train_df = interactions_df[
    interactions_df["datetime"].dt.year <= 2020
].copy()

test_df = interactions_df[
    interactions_df["datetime"].dt.year >= 2021
].copy()

print("===== TRAIN / TEST SPLIT =====")

print(
    "Training interactions:",
    len(train_df)
)

print(
    "Test interactions:",
    len(test_df)
)

print(
    "Training users:",
    train_df["user_id"].nunique()
)

print(
    "Test users:",
    test_df["user_id"].nunique()
)

print(
    "Training products:",
    train_df["parent_asin"].nunique()
)

print(
    "Test products:",
    test_df["parent_asin"].nunique()
)

NameError: name 'interactions_df' is not defined

In [48]:
# ==========================================
# RECREATE ALS USER / PRODUCT MAPPINGS
# ==========================================

import numpy as np
import pandas as pd

# Unique users and products from training data
user_ids = train_df["user_id"].unique()
product_ids = train_df["parent_asin"].unique()

# Create index mappings
user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

product_to_index = {
    product_id: index
    for index, product_id in enumerate(product_ids)
}

# Reverse product mapping
product_ids_array = np.array(product_ids)

print("===== ALS MAPPINGS =====")

print(
    "Number of users:",
    len(user_to_index)
)

print(
    "Number of products:",
    len(product_to_index)
)

print(
    "Product ID array:",
    len(product_ids_array)
)

NameError: name 'train_df' is not defined

In [49]:
from scipy.sparse import coo_matrix

# ==========================================
# CREATE USER / PRODUCT INDICES
# ==========================================

train_user_indices = train_df["user_id"].map(
    user_to_index
).to_numpy()

train_product_indices = train_df["parent_asin"].map(
    product_to_index
).to_numpy()

train_preferences = train_df["preference"].to_numpy(
    dtype=np.float32
)

# ==========================================
# CREATE SPARSE MATRIX
# ==========================================

user_item_matrix = coo_matrix(
    (
        train_preferences,
        (
            train_user_indices,
            train_product_indices
        )
    ),
    shape=(
        len(user_to_index),
        len(product_to_index)
    ),
    dtype=np.float32
).tocsr()

print("===== ALS SPARSE MATRIX =====")

print(
    "Shape:",
    user_item_matrix.shape
)

print(
    "Non-zero interactions:",
    user_item_matrix.nnz
)

print(
    "Users:",
    user_item_matrix.shape[0]
)

print(
    "Products:",
    user_item_matrix.shape[1]
)

NameError: name 'train_df' is not defined

In [50]:
from pathlib import Path

# Search your project folders for possible ALS model files
search_locations = [
    Path(r"C:\Users\amjad\Desktop\external_hdd\FurnitureRecommendationAI_Data"),
    Path(r"C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI"),
    Path(r"E:\FurnitureRecommendationAI_ImageEmbeddings")
]

possible_files = []

for folder in search_locations:
    if folder.exists():
        for pattern in [
            "*als*",
            "*ALS*",
            "*factor*",
            "*user_factors*",
            "*item_factors*",
            "*.npz"
        ]:
            possible_files.extend(folder.rglob(pattern))

# Remove duplicates
possible_files = list(dict.fromkeys(possible_files))

print("===== POSSIBLE SAVED ALS FILES =====")

if possible_files:
    for file in possible_files:
        print(file)
else:
    print("No saved ALS model/factor files found.")

===== POSSIBLE SAVED ALS FILES =====
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\narwhals
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\narwhals-2.24.0.dist-info
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\click\globals.py
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\implicit\als.py
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\implicit\approximate_als.py
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\joblib\externals
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\numpy\_globals.py
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\numpy\_g

In [51]:
from pathlib import Path

# Search only your project/data locations
search_roots = [
    Path(r"C:\Users\amjad\Desktop\external_hdd"),
    Path(r"C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI"),
    Path(r"E:\FurnitureRecommendationAI_ImageEmbeddings")
]

keywords = [
    "als",
    "factor",
    "user_factor",
    "item_factor",
    "model"
]

found = []

for root in search_roots:
    if not root.exists():
        continue

    for path in root.rglob("*"):
        if path.is_file():
            name = path.name.lower()

            if any(keyword in name for keyword in keywords):
                found.append(path)

print("===== POSSIBLE MODEL FILES =====")

if found:
    for path in found:
        print(path)
else:
    print("No ALS/model files found in project folders.")

===== POSSIBLE MODEL FILES =====
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\backend\model_loader.py
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\backend\test_model_loader.py
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\backend\__pycache__\model_loader.cpython-314.pyc
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\charset_normalizer\models.py
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\click\globals.py
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\httpcore\_models.py
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\httpx\_models.py
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\implicit\als.py
C:\Users\amjad\OneDr

In [52]:
import implicit
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

print("Implicit version:", implicit.__version__)

# ==========================================
# ALS V2 SETTINGS
# ==========================================

als_model_v2 = implicit.als.AlternatingLeastSquares(
    factors=64,
    regularization=0.1,
    iterations=20,
    random_state=42
)

print("===== TRAINING ALS V2 =====")
print("Factors: 64")
print("Regularization: 0.1")
print("Iterations: 20")
print("Matrix:", user_item_matrix.shape)

# ==========================================
# TRAIN
# ==========================================

als_model_v2.fit(
    user_item_matrix
)

print("\n===== ALS V2 TRAINING COMPLETE =====")

print(
    "User factors:",
    als_model_v2.user_factors.shape
)

print(
    "Item factors:",
    als_model_v2.item_factors.shape
)

Implicit version: 0.7.3
===== TRAINING ALS V2 =====
Factors: 64
Regularization: 0.1
Iterations: 20


NameError: name 'user_item_matrix' is not defined

In [53]:
# ==========================================
# SAVE ALS V2
# ==========================================

ALS_SAVE_DIR = Path(
    r"E:\FurnitureRecommendationAI_ImageEmbeddings\ALS_V2"
)

ALS_SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

als_model_path = (
    ALS_SAVE_DIR / "als_model_v2.npz"
)

als_model_v2.save(
    str(als_model_path)
)

print("===== ALS V2 SAVED =====")
print(als_model_path)

===== ALS V2 SAVED =====
E:\FurnitureRecommendationAI_ImageEmbeddings\ALS_V2\als_model_v2.npz


In [54]:
from sklearn.preprocessing import normalize
import numpy as np

# ==========================================
# NORMALIZE IMAGE EMBEDDINGS
# ==========================================

print("===== NORMALIZING IMAGE EMBEDDINGS =====")

# Create a normal array only for successful embeddings
successful_indices = np.where(
    image_status == 1
)[0]

successful_image_embeddings = np.asarray(
    image_embeddings[successful_indices]
)

# Normalize so dot product = cosine similarity
successful_image_embeddings = normalize(
    successful_image_embeddings,
    axis=1
)

print(
    "Successful image embeddings:",
    successful_image_embeddings.shape
)

print(
    "Products with failed images:",
    np.sum(image_status == -1)
)

print("\n✅ Image embeddings normalized!")

===== NORMALIZING IMAGE EMBEDDINGS =====
Successful image embeddings: (28203, 2048)
Products with failed images: 1797

✅ Image embeddings normalized!


In [55]:
from sklearn.preprocessing import normalize
import numpy as np

# ==========================================
# IMAGE PRODUCT LOOKUP
# ==========================================

# Map original product index → row in successful embeddings
image_index_map = {
    product_idx: image_row
    for image_row, product_idx
    in enumerate(successful_indices)
}

print("Image lookup created:")
print("Products with images:", len(image_index_map))


# ==========================================
# FUNCTION: CREATE USER IMAGE PROFILE
# ==========================================

def create_user_image_profile(user_id):

    # Get user's training interactions
    user_data = train_df[
        train_df["user_id"] == user_id
    ]

    # Keep only positive interactions
    positive_data = user_data[
        user_data["preference"] > 0
    ]

    positive_embeddings = []

    for asin in positive_data["parent_asin"]:

        # Check if product exists in image embeddings
        if asin not in product_to_index:
            continue

        product_idx = product_to_index[asin]

        if product_idx not in image_index_map:
            continue

        image_row = image_index_map[
            product_idx
        ]

        positive_embeddings.append(
            successful_image_embeddings[
                image_row
            ]
        )

    # No usable positive images
    if len(positive_embeddings) == 0:
        return None

    # Average user's liked-product images
    user_profile = np.mean(
        positive_embeddings,
        axis=0
    )

    # Normalize user profile
    user_profile = normalize(
        user_profile.reshape(1, -1),
        axis=1
    )[0]

    return user_profile


print("\n✅ User image profile function created!")

Image lookup created:
Products with images: 28203

✅ User image profile function created!


In [56]:
# ==========================================
# TEST USER IMAGE PROFILE
# ==========================================

test_user = "AE2222VGHFD37G7NOCWXASYY3ZDQ"

user_image_profile = create_user_image_profile(
    test_user
)

print("===== USER IMAGE PROFILE =====")

if user_image_profile is None:

    print(
        "No usable positive product images "
        "found for this user."
    )

else:

    print(
        "User:",
        test_user
    )

    print(
        "Profile shape:",
        user_image_profile.shape
    )

    print(
        "Profile norm:",
        np.linalg.norm(user_image_profile)
    )

    print(
        "First 10 values:",
        user_image_profile[:10]
    )

    print(
        "\n✅ User image profile created!"
    )

NameError: name 'train_df' is not defined

In [57]:
# ==========================================
# CALCULATE IMAGE SIMILARITY SCORES
# ==========================================

print("===== CALCULATING IMAGE SCORES =====")

# Calculate similarity with all successful image embeddings
image_scores = (
    successful_image_embeddings
    @ user_image_profile
)

print(
    "Number of image scores:",
    len(image_scores)
)

print(
    "Minimum score:",
    image_scores.min()
)

print(
    "Maximum score:",
    image_scores.max()
)

print(
    "Mean score:",
    image_scores.mean()
)

print(
    "\nFirst 10 image scores:",
    image_scores[:10]
)

print("\n✅ Image scores calculated!")

===== CALCULATING IMAGE SCORES =====


NameError: name 'user_image_profile' is not defined

In [58]:
# ==========================================
# MAP IMAGE SCORES TO PRODUCT IDs
# ==========================================

image_score_map = {}

# Get ASINs from the full 30,000-product catalog
catalog_asins = (
    products_df["parent_asin"]
    .astype(str)
    .to_numpy()
)

# Map successful image embeddings to their catalog products
for position, product_idx in enumerate(successful_indices):

    asin = catalog_asins[product_idx]

    image_score_map[asin] = float(
        image_scores[position]
    )

print("===== IMAGE SCORE MAPPING =====")

print(
    "Products with image scores:",
    len(image_score_map)
)

print("\nExample image scores:")

for asin, score in list(
    image_score_map.items()
)[:5]:

    print(
        asin,
        "→",
        round(score, 4)
    )

print("\n✅ Image scores mapped correctly!")

NameError: name 'image_scores' is not defined

In [59]:
# ==========================================
# 3-MODEL HYBRID SCORING
# ==========================================

ALS_WEIGHT = 0.45
TEXT_WEIGHT = 0.45
IMAGE_WEIGHT = 0.10

print("===== HYBRID MODEL WEIGHTS =====")

print("ALS   :", ALS_WEIGHT)
print("Text  :", TEXT_WEIGHT)
print("Image :", IMAGE_WEIGHT)

print(
    "\nTotal:",
    ALS_WEIGHT + TEXT_WEIGHT + IMAGE_WEIGHT
)

assert (
    ALS_WEIGHT +
    TEXT_WEIGHT +
    IMAGE_WEIGHT
) == 1.0

print("\n✅ Weights are valid!")

===== HYBRID MODEL WEIGHTS =====
ALS   : 0.45
Text  : 0.45
Image : 0.1

Total: 1.0

✅ Weights are valid!


In [60]:
print("===== CHECKING HYBRID COMPONENTS =====")

variables_to_check = [
    "als_model_v2",
    "text_embeddings",
    "image_scores",
    "image_score_map",
    "user_to_index",
    "product_to_index",
    "products_df"
]

for name in variables_to_check:
    print(
        f"{name}:",
        "AVAILABLE" if name in globals() else "NOT AVAILABLE"
    )

===== CHECKING HYBRID COMPONENTS =====
als_model_v2: AVAILABLE
text_embeddings: AVAILABLE
image_scores: NOT AVAILABLE
image_score_map: AVAILABLE
user_to_index: NOT AVAILABLE
product_to_index: NOT AVAILABLE
products_df: AVAILABLE


In [62]:
# ==========================================
# 3-MODEL HYBRID RECOMMENDATIONS
# ==========================================

test_user = "AE2222VGHFD37G7NOCWXASYY3ZDQ"

print("===== GENERATING HYBRID RECOMMENDATIONS =====")
print("User:", test_user)

# ------------------------------------------
# CHECK USER
# ------------------------------------------

if test_user not in user_to_index:

    print("❌ User not found in ALS training data.")

else:

    user_index = user_to_index[test_user]

    # --------------------------------------
    # GET ALS SCORES
    # --------------------------------------

    als_scores = (
        als_model_v2
        .user_factors[user_index]
        @
        als_model_v2.item_factors.T
    )

    # --------------------------------------
    # GET PRODUCTS
    # --------------------------------------

    candidate_asins = np.array(
        list(product_to_index.keys())
    )

    candidate_indices = np.array([
        product_to_index[asin]
        for asin in candidate_asins
    ])

    # --------------------------------------
    # GET TEXT SCORES
    # --------------------------------------

    # Get user's positive products
    user_data = train_df[
        train_df["user_id"] == test_user
    ]

    positive_data = user_data[
        user_data["preference"] > 0
    ]

    positive_asins = (
        positive_data["parent_asin"]
        .tolist()
    )

    positive_text_embeddings = []

    for asin in positive_asins:

        if asin in product_to_index:

            idx = product_to_index[asin]

            positive_text_embeddings.append(
                text_embeddings[idx]
            )

    if positive_text_embeddings:

        user_text_profile = np.mean(
            positive_text_embeddings,
            axis=0
        )

        user_text_profile = (
            user_text_profile /
            (np.linalg.norm(user_text_profile) + 1e-8)
        )

        text_scores = (
            text_embeddings[candidate_indices]
            @
            user_text_profile
        )

    else:

        text_scores = np.zeros(
            len(candidate_asins)
        )

    # --------------------------------------
    # GET IMAGE SCORES
    # --------------------------------------

    image_scores_for_candidates = np.array([
        image_score_map.get(
            asin,
            0.0
        )
        for asin in candidate_asins
    ])

    # --------------------------------------
    # NORMALIZE ALL SCORES
    # --------------------------------------

    def minmax_normalize(scores):

        min_value = scores.min()
        max_value = scores.max()

        if max_value == min_value:
            return np.zeros_like(
                scores,
                dtype=np.float32
            )

        return (
            (scores - min_value) /
            (max_value - min_value)
        )

    als_norm = minmax_normalize(
        als_scores[candidate_indices]
    )

    text_norm = minmax_normalize(
        text_scores
    )

    image_norm = minmax_normalize(
        image_scores_for_candidates
    )

    # --------------------------------------
    # HYBRID SCORE
    # --------------------------------------

    hybrid_scores = (
        ALS_WEIGHT * als_norm +
        TEXT_WEIGHT * text_norm +
        IMAGE_WEIGHT * image_norm
    )

    # --------------------------------------
    # REMOVE PRODUCTS ALREADY INTERACTED
    # --------------------------------------

    already_seen = set(
        user_data["parent_asin"]
    )

    for i, asin in enumerate(candidate_asins):

        if asin in already_seen:
            hybrid_scores[i] = -np.inf

    # --------------------------------------
    # TOP 10
    # --------------------------------------

    top_indices = np.argsort(
        -hybrid_scores
    )[:10]

    print("\n===== HYBRID TOP 10 =====")

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        asin = candidate_asins[idx]

        product_rows = products_df[
            products_df["parent_asin"] == asin
        ]

        if len(product_rows) > 0:

            title = product_rows.iloc[0].get(
                "title",
                "Unknown product"
            )

        else:

            title = "Unknown product"

        print(
            f"\n{rank}. {title}"
        )

        print(
            "   ASIN:",
            asin
        )

        print(
            "   ALS:",
            round(
                float(als_norm[idx]),
                4
            )
        )

        print(
            "   Text:",
            round(
                float(text_norm[idx]),
                4
            )
        )

        print(
            "   Image:",
            round(
                float(image_norm[idx]),
                4
            )
        )

        print(
            "   Hybrid:",
            round(
                float(hybrid_scores[idx]),
                4
            )
        )

    print(
        "\n✅ 3-model hybrid recommendations generated!"
    )

===== GENERATING HYBRID RECOMMENDATIONS =====
User: AE2222VGHFD37G7NOCWXASYY3ZDQ


NameError: name 'user_to_index' is not defined

In [63]:
# ==========================================
# FAST 3-MODEL HYBRID EVALUATION
# ==========================================

import numpy as np
from tqdm.auto import tqdm

TOP_K = 10

ALS_WEIGHT = 0.50
TEXT_WEIGHT = 0.45
IMAGE_WEIGHT = 0.05

# ------------------------------------------
# 1. PREPARE PRODUCT INDICES
# ------------------------------------------

# Full 30,000-product catalog
catalog_asins = (
    products_df["parent_asin"]
    .astype(str)
    .to_numpy()
)

catalog_asin_to_index = {
    asin: i
    for i, asin in enumerate(catalog_asins)
}

# ALS products → full catalog indices
als_asins = np.asarray(
    list(product_to_index.keys())
)

als_full_indices = np.array([
    catalog_asin_to_index[asin]
    for asin in als_asins
])


# ------------------------------------------
# 2. PREPARE NORMALIZED TEXT EMBEDDINGS
# ------------------------------------------

text_norm_all = text_embeddings.astype(
    np.float32,
    copy=True
)

text_norm_all /= (
    np.linalg.norm(
        text_norm_all,
        axis=1,
        keepdims=True
    ) + 1e-8
)

# Text embeddings corresponding to ALS products
text_als = text_norm_all[
    als_full_indices
]


# ------------------------------------------
# 3. PREPARE IMAGE EMBEDDINGS
# ------------------------------------------

# successful_indices are indices in the
# full 30,000-product catalog.

image_full_index_to_row = {
    int(product_idx): row
    for row, product_idx
    in enumerate(successful_indices)
}

# Image embeddings are already normalized
image_matrix = successful_image_embeddings.astype(
    np.float32,
    copy=False
)


# ------------------------------------------
# 4. CREATE FAST USER → POSITIVE PRODUCTS
# ------------------------------------------

positive_train = train_df[
    train_df["preference"] > 0
]

user_positive = (
    positive_train
    .groupby("user_id")["parent_asin"]
    .apply(list)
    .to_dict()
)

positive_test = test_df[
    test_df["preference"] > 0
]

user_test_positive = (
    positive_test
    .groupby("user_id")["parent_asin"]
    .apply(set)
    .to_dict()
)


# ------------------------------------------
# 5. SELECT 1000 EVALUATION USERS
# ------------------------------------------

eligible_users = [
    user_id
    for user_id in user_test_positive
    if (
        user_id in user_to_index
        and user_id in user_positive
    )
]

rng = np.random.default_rng(42)

sample_size = min(
    1000,
    len(eligible_users)
)

eval_users = rng.choice(
    eligible_users,
    size=sample_size,
    replace=False
)

print("===== FAST HYBRID EVALUATION =====")
print(
    "Evaluation users:",
    len(eval_users)
)


# ------------------------------------------
# 6. PRE-CALCULATE PRODUCT FACTORS
# ------------------------------------------

item_factors = (
    als_model_v2.item_factors
)

user_factors = (
    als_model_v2.user_factors
)


# ------------------------------------------
# 7. METRIC STORAGE
# ------------------------------------------

recall_scores = []
hit_scores = []
ndcg_scores = []

successful_users = 0


# ------------------------------------------
# 8. EVALUATION
# ------------------------------------------

for user_id in tqdm(
    eval_users,
    desc="Evaluating 3-model Hybrid"
):

    # ======================================
    # USER INDEX
    # ======================================

    user_idx = user_to_index[user_id]

    # ======================================
    # ALS SCORES
    # ======================================

    als_scores = (
        user_factors[user_idx]
        @
        item_factors.T
    )

    # ======================================
    # TEXT PROFILE
    # ======================================

    positive_asins = user_positive[user_id]

    positive_text_indices = [
        catalog_asin_to_index[asin]
        for asin in positive_asins
        if asin in catalog_asin_to_index
    ]

    if len(positive_text_indices) == 0:
        continue

    text_profile = np.mean(
        text_norm_all[
            positive_text_indices
        ],
        axis=0
    )

    text_profile /= (
        np.linalg.norm(text_profile)
        + 1e-8
    )

    text_scores = (
        text_als
        @
        text_profile
    )

    # ======================================
    # IMAGE PROFILE
    # ======================================

    positive_image_rows = []

    for full_idx in positive_text_indices:

        if full_idx in image_full_index_to_row:

            positive_image_rows.append(
                image_full_index_to_row[
                    full_idx
                ]
            )

    if len(positive_image_rows) == 0:
        continue

    image_profile = np.mean(
        image_matrix[
            positive_image_rows
        ],
        axis=0
    )

    image_profile /= (
        np.linalg.norm(image_profile)
        + 1e-8
    )

    # Image scores for products with images
    image_scores_successful = (
        image_matrix
        @
        image_profile
    )

    # Map image scores to ALS products
    image_scores = np.zeros(
        len(als_asins),
        dtype=np.float32
    )

    for als_idx, full_idx in enumerate(
        als_full_indices
    ):

        if full_idx in image_full_index_to_row:

            image_row = (
                image_full_index_to_row[
                    full_idx
                ]
            )

            image_scores[als_idx] = (
                image_scores_successful[
                    image_row
                ]
            )

    # ======================================
    # MIN-MAX NORMALIZATION
    # ======================================

    def minmax(x):

        minimum = x.min()
        maximum = x.max()

        if maximum == minimum:
            return np.zeros_like(x)

        return (
            (x - minimum)
            / (maximum - minimum)
        )

    als_norm = minmax(
        als_scores
    )

    text_norm = minmax(
        text_scores
    )

    image_norm = minmax(
        image_scores
    )

    # ======================================
    # HYBRID SCORE
    # ======================================

    hybrid_scores = (
        ALS_WEIGHT * als_norm
        +
        TEXT_WEIGHT * text_norm
        +
        IMAGE_WEIGHT * image_norm
    )

    # ======================================
    # REMOVE ALREADY-SEEN PRODUCTS
    # ======================================

    seen = set(
        positive_train[
            positive_train["user_id"] == user_id
        ]["parent_asin"]
    )

    for idx, asin in enumerate(als_asins):

        if asin in seen:
            hybrid_scores[idx] = -np.inf

    # ======================================
    # TOP 10
    # ======================================

    top_indices = np.argpartition(
        -hybrid_scores,
        TOP_K
    )[:TOP_K]

    # Sort the selected 10
    top_indices = top_indices[
        np.argsort(
            -hybrid_scores[top_indices]
        )
    ]

    recommended = set(
        als_asins[top_indices]
    )

    # ======================================
    # TEST POSITIVES
    # ======================================

    test_positive = (
        user_test_positive[user_id]
        &
        set(als_asins)
    )

    if len(test_positive) == 0:
        continue

    # ======================================
    # HITS
    # ======================================

    hits = (
        recommended
        &
        test_positive
    )

    hit_count = len(hits)

    # ======================================
    # RECALL@10
    # ======================================

    recall = (
        hit_count /
        len(test_positive)
    )

    recall_scores.append(recall)

    # ======================================
    # HIT RATE@10
    # ======================================

    hit_scores.append(
        1.0 if hit_count > 0 else 0.0
    )

    # ======================================
    # NDCG@10
    # ======================================

    dcg = 0.0

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        if als_asins[idx] in test_positive:

            dcg += (
                1 /
                np.log2(rank + 1)
            )

    ideal_hits = min(
        len(test_positive),
        TOP_K
    )

    idcg = sum(
        1 /
        np.log2(rank + 1)
        for rank in range(
            1,
            ideal_hits + 1
        )
    )

    ndcg = (
        dcg / idcg
        if idcg > 0
        else 0.0
    )

    ndcg_scores.append(ndcg)

    successful_users += 1


# ==========================================
# FINAL RESULTS
# ==========================================

print("\n========================================")
print("3-MODEL HYBRID EVALUATION")
print("========================================")

print(
    "ALS weight:",
    ALS_WEIGHT
)

print(
    "Text weight:",
    TEXT_WEIGHT
)

print(
    "Image weight:",
    IMAGE_WEIGHT
)

print(
    "Recall@10:",
    round(
        np.mean(recall_scores),
        4
    )
)

print(
    "Hit Rate@10:",
    round(
        np.mean(hit_scores),
        4
    )
)

print(
    "NDCG@10:",
    round(
        np.mean(ndcg_scores),
        4
    )
)

print(
    "Users successfully evaluated:",
    successful_users
)

NameError: name 'product_to_index' is not defined

In [64]:
# ==========================================
# FAST FIXED EVALUATION USER SET
# ==========================================

print("Creating fast user lookup tables...")

# Positive training interactions
positive_train_df = train_df[
    train_df["preference"] > 0
]

# Positive test interactions
positive_test_df = test_df[
    test_df["preference"] > 0
]

# Users with positive training interactions
train_positive_users = set(
    positive_train_df["user_id"].unique()
)

# Users with positive test interactions
test_positive_users = set(
    positive_test_df["user_id"].unique()
)

# Users known to ALS
als_users = set(
    user_to_index.keys()
)

# Intersection
eligible_users = list(
    train_positive_users
    &
    test_positive_users
    &
    als_users
)

print("===== FAST EVALUATION USERS =====")

print(
    "Positive training users:",
    len(train_positive_users)
)

print(
    "Positive test users:",
    len(test_positive_users)
)

print(
    "ALS users:",
    len(als_users)
)

print(
    "Eligible users:",
    len(eligible_users)
)


# ==========================================
# SELECT FIXED 1000 USERS
# ==========================================

rng = np.random.default_rng(42)

evaluation_user_count = min(
    1000,
    len(eligible_users)
)

evaluation_users = list(
    rng.choice(
        eligible_users,
        size=evaluation_user_count,
        replace=False
    )
)

print(
    "\nSelected users:",
    len(evaluation_users)
)

print(
    "Random seed: 42"
)

print(
    "\n✅ Fixed evaluation set created quickly!"
)

Creating fast user lookup tables...


NameError: name 'train_df' is not defined

In [65]:
# ==========================================
# FAST USER INTERACTION LOOKUPS
# ==========================================

print("Creating fast interaction lookups...")

# Positive training products for each user
train_positive_lookup = (
    positive_train_df
    .groupby("user_id")["parent_asin"]
    .apply(set)
    .to_dict()
)

# Positive test products for each user
test_positive_lookup = (
    positive_test_df
    .groupby("user_id")["parent_asin"]
    .apply(set)
    .to_dict()
)

print("Training lookup users:", len(train_positive_lookup))
print("Test lookup users:", len(test_positive_lookup))

print("\n✅ Fast interaction lookups created!")

Creating fast interaction lookups...


NameError: name 'positive_train_df' is not defined

In [66]:
# ==========================================
# FAIR COMPARISON OF HYBRID MODELS
# ==========================================

import numpy as np
from tqdm.auto import tqdm

TOP_K = 10

# ------------------------------------------
# Model configurations
# ------------------------------------------

model_configs = {
    "ALS + Text (50/50)": (0.50, 0.50, 0.00),
    "ALS + Text + Image (50/45/5)": (0.50, 0.45, 0.05),
    "ALS + Text + Image (45/45/10)": (0.45, 0.45, 0.10),
    "ALS + Text + Image (50/30/20)": (0.50, 0.30, 0.20),
}

# ------------------------------------------
# Storage
# ------------------------------------------

results = {}

for model_name in model_configs:

    results[model_name] = {
        "recall": [],
        "hit": [],
        "ndcg": []
    }


# ------------------------------------------
# Evaluate each user
# ------------------------------------------

for user_id in tqdm(
    evaluation_users,
    desc="Fair evaluation"
):

    if user_id not in train_positive_lookup:
        continue

    if user_id not in test_positive_lookup:
        continue

    user_idx = user_to_index[user_id]

    # ======================================
    # ALS SCORES
    # ======================================

    als_scores = (
        als_model_v2.user_factors[user_idx]
        @
        als_model_v2.item_factors.T
    )

    # ======================================
    # TEXT PROFILE
    # ======================================

    positive_asins = train_positive_lookup[
        user_id
    ]

    text_indices = [
        catalog_asin_to_index[asin]
        for asin in positive_asins
        if asin in catalog_asin_to_index
    ]

    if len(text_indices) == 0:
        continue

    text_profile = np.mean(
        text_norm_all[text_indices],
        axis=0
    )

    text_profile /= (
        np.linalg.norm(text_profile)
        + 1e-8
    )

    text_scores = (
        text_als @ text_profile
    )

    # ======================================
    # IMAGE PROFILE
    # ======================================

    image_profile = create_user_image_profile(
        user_id
    )

    if image_profile is None:
        image_scores = np.zeros(
            len(als_asins),
            dtype=np.float32
        )

    else:

        image_scores_successful = (
            image_matrix @ image_profile
        )

        image_scores = np.zeros(
            len(als_asins),
            dtype=np.float32
        )

        for als_idx, full_idx in enumerate(
            als_full_indices
        ):

            image_row = (
                image_full_index_to_row.get(
                    int(full_idx)
                )
            )

            if image_row is not None:

                image_scores[als_idx] = (
                    image_scores_successful[
                        image_row
                    ]
                )

    # ======================================
    # NORMALIZE
    # ======================================

    def minmax(x):

        minimum = x.min()
        maximum = x.max()

        if maximum == minimum:
            return np.zeros_like(x)

        return (
            (x - minimum)
            / (maximum - minimum)
        )

    als_norm = minmax(als_scores)

    text_norm_scores = minmax(
        text_scores
    )

    image_norm_scores = minmax(
        image_scores
    )

    # ======================================
    # ALREADY SEEN PRODUCTS
    # ======================================

    seen_products = train_positive_lookup[
        user_id
    ]

    # ======================================
    # TEST PRODUCTS
    # ======================================

    test_products = (
        test_positive_lookup[user_id]
        &
        set(als_asins)
    )

    if len(test_products) == 0:
        continue

    # ======================================
    # EVALUATE EACH MODEL
    # ======================================

    for model_name, weights in model_configs.items():

        als_w, text_w, image_w = weights

        hybrid_scores = (
            als_w * als_norm
            +
            text_w * text_norm_scores
            +
            image_w * image_norm_scores
        )

        # Remove products already seen
        hybrid_scores = hybrid_scores.copy()

        for idx, asin in enumerate(als_asins):

            if asin in seen_products:

                hybrid_scores[idx] = -np.inf

        # Top K
        top_indices = np.argpartition(
            -hybrid_scores,
            TOP_K
        )[:TOP_K]

        top_indices = top_indices[
            np.argsort(
                -hybrid_scores[top_indices]
            )
        ]

        recommended = set(
            als_asins[top_indices]
        )

        # ==================================
        # HITS
        # ==================================

        hits = (
            recommended
            &
            test_products
        )

        hit_count = len(hits)

        # ==================================
        # RECALL
        # ==================================

        recall = (
            hit_count /
            len(test_products)
        )

        # ==================================
        # HIT RATE
        # ==================================

        hit_rate = (
            1.0
            if hit_count > 0
            else 0.0
        )

        # ==================================
        # NDCG
        # ==================================

        dcg = 0.0

        for rank, idx in enumerate(
            top_indices,
            start=1
        ):

            if (
                als_asins[idx]
                in test_products
            ):

                dcg += (
                    1 /
                    np.log2(rank + 1)
                )

        ideal_hits = min(
            len(test_products),
            TOP_K
        )

        idcg = sum(
            1 /
            np.log2(rank + 1)
            for rank in range(
                1,
                ideal_hits + 1
            )
        )

        ndcg = (
            dcg / idcg
            if idcg > 0
            else 0.0
        )

        results[model_name][
            "recall"
        ].append(recall)

        results[model_name][
            "hit"
        ].append(hit_rate)

        results[model_name][
            "ndcg"
        ].append(ndcg)


# ==========================================
# FINAL COMPARISON
# ==========================================

print("\n")
print("=" * 75)
print("FINAL FAIR MODEL COMPARISON")
print("=" * 75)

for model_name, metrics in results.items():

    print("\n", model_name)

    print(
        "Recall@10:",
        round(
            np.mean(metrics["recall"]),
            4
        )
    )

    print(
        "Hit Rate@10:",
        round(
            np.mean(metrics["hit"]),
            4
        )
    )

    print(
        "NDCG@10:",
        round(
            np.mean(metrics["ndcg"]),
            4
        )
    )

    print(
        "Users:",
        len(metrics["recall"])
    )

print("\n✅ FAIR COMPARISON COMPLETE!")

NameError: name 'evaluation_users' is not defined

In [67]:
# ==========================================
# SHOW COMPLETE FAIR RESULTS
# ==========================================

print("=" * 75)
print("COMPLETE FAIR MODEL COMPARISON")
print("=" * 75)

for model_name, metrics in results.items():

    print(f"\n{model_name}")

    print(
        "Recall@10:",
        round(np.mean(metrics["recall"]), 6)
    )

    print(
        "Hit Rate@10:",
        round(np.mean(metrics["hit"]), 6)
    )

    print(
        "NDCG@10:",
        round(np.mean(metrics["ndcg"]), 6)
    )

    print(
        "Users:",
        len(metrics["recall"])
    )

COMPLETE FAIR MODEL COMPARISON

ALS + Text (50/50)
Recall@10: nan
Hit Rate@10: nan
NDCG@10: nan
Users: 0

ALS + Text + Image (50/45/5)
Recall@10: nan
Hit Rate@10: nan
NDCG@10: nan
Users: 0

ALS + Text + Image (45/45/10)
Recall@10: nan
Hit Rate@10: nan
NDCG@10: nan
Users: 0

ALS + Text + Image (50/30/20)
Recall@10: nan
Hit Rate@10: nan
NDCG@10: nan
Users: 0


c:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\numpy\_core\fromnumeric.py:3862: RuntimeWarning: Mean of empty slice
  return _methods._mean(a, axis=axis, dtype=dtype,
c:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\.venv-1\Lib\site-packages\numpy\_core\_methods.py:142: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [68]:
# ==========================================
# BASELINE EVALUATION DIAGNOSTIC
# ==========================================

print("===== BASELINE DATA CHECK =====")

print("Fixed evaluation users:", len(evaluation_users))

print(
    "Users in training lookup:",
    len(train_positive_lookup)
)

print(
    "Users in test lookup:",
    len(test_positive_lookup)
)

print(
    "ALS users:",
    len(user_to_index)
)

# Check first 5 users
print("\n===== FIRST 5 USERS =====")

for user_id in evaluation_users[:5]:

    train_items = train_positive_lookup.get(
        user_id,
        set()
    )

    test_items = test_positive_lookup.get(
        user_id,
        set()
    )

    als_exists = user_id in user_to_index

    print("\nUser:", user_id)
    print("Training positives:", len(train_items))
    print("Test positives:", len(test_items))
    print("ALS available:", als_exists)

print("\n✅ Diagnostic complete!")

===== BASELINE DATA CHECK =====


NameError: name 'evaluation_users' is not defined

In [69]:
# ==========================================
# SINGLE USER BASELINE CHECK
# ==========================================

check_user = evaluation_users[0]

print("===== SINGLE USER BASELINE CHECK =====")
print("User:", check_user)

# ------------------------------------------
# Training products
# ------------------------------------------

train_products = train_positive_lookup.get(
    check_user,
    set()
)

print("\nTraining positive products:")
for asin in list(train_products)[:10]:
    print("-", asin)

# ------------------------------------------
# Test products
# ------------------------------------------

test_products = test_positive_lookup.get(
    check_user,
    set()
)

print("\nTest positive products:")
for asin in test_products:
    print("-", asin)

# ------------------------------------------
# ALS user
# ------------------------------------------

user_idx = user_to_index[check_user]

als_scores = (
    als_model_v2.user_factors[user_idx]
    @
    als_model_v2.item_factors.T
)

# ------------------------------------------
# Remove training products
# ------------------------------------------

als_scores_clean = als_scores.copy()

for idx, asin in enumerate(als_asins):

    if asin in train_products:
        als_scores_clean[idx] = -np.inf

# ------------------------------------------
# Top 10 ALS recommendations
# ------------------------------------------

top_indices = np.argsort(
    -als_scores_clean
)[:10]

print("\n===== ALS TOP 10 =====")

hits = 0

for rank, idx in enumerate(
    top_indices,
    start=1
):

    asin = als_asins[idx]

    is_test_positive = (
        asin in test_products
    )

    if is_test_positive:
        hits += 1

    print(
        f"{rank}. {asin}"
        f" | Score: {als_scores_clean[idx]:.4f}"
        f" | Test positive: {is_test_positive}"
    )

print("\nTest positives:", len(test_products))
print("Hits in Top 10:", hits)

print("\n✅ Single-user baseline check complete!")

NameError: name 'evaluation_users' is not defined

In [70]:
# ==========================================
# CLEAN BASELINE: ALS + MINILM
# ==========================================

import numpy as np
from tqdm.auto import tqdm

TOP_K = 10

ALS_WEIGHT = 0.50
TEXT_WEIGHT = 0.50

recall_scores = []
hit_scores = []
ndcg_scores = []

successful_users = 0

print("===== CLEAN ALS + MINILM BASELINE =====")
print("Users to evaluate:", len(evaluation_users))
print("ALS weight:", ALS_WEIGHT)
print("Text weight:", TEXT_WEIGHT)


# ==========================================
# EVALUATION
# ==========================================

for user_id in tqdm(
    evaluation_users,
    desc="Evaluating ALS + MiniLM"
):

    # --------------------------------------
    # USER INDEX
    # --------------------------------------

    user_idx = user_to_index[user_id]

    # --------------------------------------
    # ALS SCORES
    # --------------------------------------

    als_scores = (
        als_model_v2.user_factors[user_idx]
        @
        als_model_v2.item_factors.T
    )

    # --------------------------------------
    # USER'S POSITIVE TRAINING PRODUCTS
    # --------------------------------------

    train_products = train_positive_lookup.get(
        user_id,
        set()
    )

    # --------------------------------------
    # USER'S POSITIVE TEST PRODUCTS
    # --------------------------------------

    test_products = test_positive_lookup.get(
        user_id,
        set()
    )

    # Only products available to ALS
    test_products = (
        test_products
        &
        set(als_asins)
    )

    if len(test_products) == 0:
        continue

    # --------------------------------------
    # TEXT PROFILE
    # --------------------------------------

    text_indices = [
        catalog_asin_to_index[asin]
        for asin in train_products
        if asin in catalog_asin_to_index
    ]

    if len(text_indices) == 0:
        continue

    user_text_profile = np.mean(
        text_norm_all[text_indices],
        axis=0
    )

    user_text_profile /= (
        np.linalg.norm(
            user_text_profile
        ) + 1e-8
    )

    # --------------------------------------
    # TEXT SCORES
    # --------------------------------------

    text_scores = (
        text_als
        @
        user_text_profile
    )

    # --------------------------------------
    # NORMALIZE
    # --------------------------------------

    def minmax_score(x):

        minimum = x.min()
        maximum = x.max()

        if maximum == minimum:
            return np.zeros_like(x)

        return (
            (x - minimum)
            /
            (maximum - minimum)
        )

    als_norm = minmax_score(
        als_scores
    )

    text_norm_scores = minmax_score(
        text_scores
    )

    # --------------------------------------
    # COMBINE ALS + TEXT
    # --------------------------------------

    final_scores = (
        ALS_WEIGHT * als_norm
        +
        TEXT_WEIGHT * text_norm_scores
    )

    # --------------------------------------
    # REMOVE TRAINING PRODUCTS
    # --------------------------------------

    final_scores = final_scores.copy()

    for idx, asin in enumerate(als_asins):

        if asin in train_products:
            final_scores[idx] = -np.inf

    # --------------------------------------
    # TOP 10
    # --------------------------------------

    top_indices = np.argpartition(
        -final_scores,
        TOP_K
    )[:TOP_K]

    top_indices = top_indices[
        np.argsort(
            -final_scores[top_indices]
        )
    ]

    recommended = set(
        als_asins[top_indices]
    )

    # --------------------------------------
    # HITS
    # --------------------------------------

    hits = (
        recommended
        &
        test_products
    )

    hit_count = len(hits)

    # --------------------------------------
    # RECALL@10
    # --------------------------------------

    recall = (
        hit_count
        /
        len(test_products)
    )

    # --------------------------------------
    # HIT RATE@10
    # --------------------------------------

    hit_rate = (
        1.0
        if hit_count > 0
        else 0.0
    )

    # --------------------------------------
    # NDCG@10
    # --------------------------------------

    dcg = 0.0

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        if als_asins[idx] in test_products:

            dcg += (
                1
                /
                np.log2(rank + 1)
            )

    ideal_hits = min(
        len(test_products),
        TOP_K
    )

    idcg = sum(
        1 /
        np.log2(rank + 1)
        for rank in range(
            1,
            ideal_hits + 1
        )
    )

    ndcg = (
        dcg / idcg
        if idcg > 0
        else 0.0
    )

    # --------------------------------------
    # STORE RESULTS
    # --------------------------------------

    recall_scores.append(recall)
    hit_scores.append(hit_rate)
    ndcg_scores.append(ndcg)

    successful_users += 1


# ==========================================
# FINAL RESULTS
# ==========================================

print("\n")
print("=" * 60)
print("CLEAN ALS + MINILM BASELINE RESULTS")
print("=" * 60)

print(
    "Recall@10:",
    round(
        np.mean(recall_scores),
        6
    )
)

print(
    "Hit Rate@10:",
    round(
        np.mean(hit_scores),
        6
    )
)

print(
    "NDCG@10:",
    round(
        np.mean(ndcg_scores),
        6
    )
)

print(
    "Users successfully evaluated:",
    successful_users
)

print("\n✅ Clean baseline evaluation complete!")

===== CLEAN ALS + MINILM BASELINE =====


NameError: name 'evaluation_users' is not defined

In [71]:
# ==========================================
# FINAL HYBRID RECOMMENDER FUNCTION
# ==========================================

def recommend_products(
    user_id,
    top_k=10,
    als_weight=0.45,
    text_weight=0.45,
    image_weight=0.10
):

    if user_id not in user_to_index:
        print("❌ User not found in ALS model.")
        return None

    # --------------------------------------
    # USER INFORMATION
    # --------------------------------------

    user_idx = user_to_index[user_id]

    train_products = train_positive_lookup.get(
        user_id,
        set()
    )

    # --------------------------------------
    # ALS SCORES
    # --------------------------------------

    als_scores = (
        als_model_v2.user_factors[user_idx]
        @ als_model_v2.item_factors.T
    )

    # --------------------------------------
    # TEXT PROFILE
    # --------------------------------------

    text_indices = [
        catalog_asin_to_index[asin]
        for asin in train_products
        if asin in catalog_asin_to_index
    ]

    if len(text_indices) > 0:

        text_profile = np.mean(
            text_norm_all[text_indices],
            axis=0
        )

        text_profile /= (
            np.linalg.norm(text_profile)
            + 1e-8
        )

        text_scores = (
            text_als @ text_profile
        )

    else:

        text_scores = np.zeros(
            len(als_asins)
        )

    # --------------------------------------
    # IMAGE PROFILE
    # --------------------------------------

    image_profile = create_user_image_profile(
        user_id
    )

    image_scores = np.zeros(
        len(als_asins),
        dtype=np.float32
    )

    if image_profile is not None:

        image_scores_successful = (
            image_matrix @ image_profile
        )

        for als_idx, full_idx in enumerate(
            als_full_indices
        ):

            image_row = (
                image_full_index_to_row.get(
                    int(full_idx)
                )
            )

            if image_row is not None:

                image_scores[als_idx] = (
                    image_scores_successful[
                        image_row
                    ]
                )

    # --------------------------------------
    # NORMALIZATION
    # --------------------------------------

    def minmax(x):

        minimum = x.min()
        maximum = x.max()

        if maximum == minimum:
            return np.zeros_like(x)

        return (
            (x - minimum)
            /
            (maximum - minimum)
        )

    als_norm = minmax(als_scores)

    text_norm_scores = minmax(
        text_scores
    )

    image_norm_scores = minmax(
        image_scores
    )

    # --------------------------------------
    # FINAL HYBRID SCORE
    # --------------------------------------

    final_scores = (
        als_weight * als_norm
        +
        text_weight * text_norm_scores
        +
        image_weight * image_norm_scores
    )

    # --------------------------------------
    # REMOVE ALREADY SEEN PRODUCTS
    # --------------------------------------

    final_scores = final_scores.copy()

    for idx, asin in enumerate(als_asins):

        if asin in train_products:
            final_scores[idx] = -np.inf

    # --------------------------------------
    # TOP K
    # --------------------------------------

    top_indices = np.argsort(
        -final_scores
    )[:top_k]

    # --------------------------------------
    # CREATE RESULT
    # --------------------------------------

    recommendations = []

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        asin = als_asins[idx]

        product_rows = products_df[
            products_df["parent_asin"] == asin
        ]

        if len(product_rows) > 0:

            row = product_rows.iloc[0]

            title = row.get(
                "title",
                "Unknown product"
            )

            image_url = row.get(
                "image_url",
                None
            )

        else:

            title = "Unknown product"
            image_url = None

        recommendations.append({
            "rank": rank,
            "parent_asin": asin,
            "title": title,
            "als_score": float(
                als_norm[idx]
            ),
            "text_score": float(
                text_norm_scores[idx]
            ),
            "image_score": float(
                image_norm[idx]
            ),
            "hybrid_score": float(
                final_scores[idx]
            ),
            "image_url": image_url
        })

    return pd.DataFrame(
        recommendations
    )

In [72]:
# ==========================================
# TEST FINAL RECOMMENDER
# ==========================================

final_recommendations = recommend_products(
    user_id="AE2222VGHFD37G7NOCWXASYY3ZDQ",
    top_k=10,
    als_weight=0.45,
    text_weight=0.45,
    image_weight=0.10
)

print("===== FINAL TOP 10 RECOMMENDATIONS =====")

display(
    final_recommendations[
        [
            "rank",
            "title",
            "parent_asin",
            "als_score",
            "text_score",
            "image_score",
            "hybrid_score"
        ]
    ]
)

NameError: name 'user_to_index' is not defined

In [73]:
# ==========================================
# FINAL MODEL COMPONENT CHECK
# ==========================================

print("===== FINAL MODEL COMPONENTS =====")

components = [
    "als_model_v2",
    "text_embeddings",
    "image_embeddings",
    "successful_image_embeddings",
    "products_df",
    "train_df",
    "test_df",
    "user_to_index",
    "product_to_index",
    "product_ids_array",
    "catalog_asins",
    "catalog_asin_to_index",
    "text_norm_all",
    "text_als",
    "image_full_index_to_row",
    "train_positive_lookup",
    "test_positive_lookup",
    "evaluation_users"
]

for component in components:

    if component in globals():

        obj = globals()[component]

        try:
            shape = obj.shape
        except:
            shape = "N/A"

        print(
            f"✅ {component:30} Shape: {shape}"
        )

    else:

        print(
            f"❌ {component:30} NOT AVAILABLE"
        )

print("\n✅ Component check complete!")

===== FINAL MODEL COMPONENTS =====
✅ als_model_v2                   Shape: N/A
✅ text_embeddings                Shape: (30000, 384)
✅ image_embeddings               Shape: (30000, 2048)
✅ successful_image_embeddings    Shape: (28203, 2048)
✅ products_df                    Shape: (30000, 9)
❌ train_df                       NOT AVAILABLE
❌ test_df                        NOT AVAILABLE
❌ user_to_index                  NOT AVAILABLE
❌ product_to_index               NOT AVAILABLE
❌ product_ids_array              NOT AVAILABLE
✅ catalog_asins                  Shape: (30000,)
✅ catalog_asin_to_index          Shape: N/A
❌ text_norm_all                  NOT AVAILABLE
❌ text_als                       NOT AVAILABLE
❌ image_full_index_to_row        NOT AVAILABLE
❌ train_positive_lookup          NOT AVAILABLE
❌ test_positive_lookup           NOT AVAILABLE
❌ evaluation_users               NOT AVAILABLE

✅ Component check complete!


In [74]:
# ==========================================
# SAVE FINAL RECOMMENDATION COMPONENTS
# ==========================================

from pathlib import Path
import numpy as np
import pandas as pd
import pickle

SAVE_DIR = Path(
    r"E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL"
)

SAVE_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("===== SAVING FINAL MODEL COMPONENTS =====")


# ==========================================
# 1. SAVE ALS MODEL
# ==========================================

als_save_path = SAVE_DIR / "als_model_v2.npz"

als_model_v2.save(
    str(als_save_path)
)

print("✅ ALS model saved")


# ==========================================
# 2. SAVE TEXT EMBEDDINGS
# ==========================================

np.save(
    SAVE_DIR / "text_embeddings.npy",
    text_embeddings
)

print("✅ Text embeddings saved")


# ==========================================
# 3. SAVE IMAGE EMBEDDINGS
# ==========================================

# The original image_embeddings are already
# stored on E: but we save a reference/status
# file here.

np.save(
    SAVE_DIR / "successful_indices.npy",
    successful_indices
)

print("✅ Image index information saved")


# ==========================================
# 4. SAVE PRODUCT DATA
# ==========================================

products_df.to_parquet(
    SAVE_DIR / "products.parquet",
    index=False
)

print("✅ Product data saved")


# ==========================================
# 5. SAVE ALS MAPPINGS
# ==========================================

with open(
    SAVE_DIR / "user_to_index.pkl",
    "wb"
) as f:

    pickle.dump(
        user_to_index,
        f
    )


with open(
    SAVE_DIR / "product_to_index.pkl",
    "wb"
) as f:

    pickle.dump(
        product_to_index,
        f
    )


np.save(
    SAVE_DIR / "product_ids_array.npy",
    product_ids_array
)

print("✅ ALS mappings saved")


# ==========================================
# 6. SAVE CATALOG MAPPING
# ==========================================

with open(
    SAVE_DIR / "catalog_asin_to_index.pkl",
    "wb"
) as f:

    pickle.dump(
        catalog_asin_to_index,
        f
    )

np.save(
    SAVE_DIR / "catalog_asins.npy",
    catalog_asins
)

print("✅ Catalog mappings saved")


# ==========================================
# 7. SAVE TEXT NORMALIZATION
# ==========================================

np.save(
    SAVE_DIR / "text_norm_all.npy",
    text_norm_all
)

np.save(
    SAVE_DIR / "text_als.npy",
    text_als
)

print("✅ Text normalization data saved")


# ==========================================
# 8. SAVE IMAGE LOOKUP
# ==========================================

with open(
    SAVE_DIR / "image_full_index_to_row.pkl",
    "wb"
) as f:

    pickle.dump(
        image_full_index_to_row,
        f
    )

print("✅ Image lookup saved")


# ==========================================
# 9. SAVE HYBRID SETTINGS
# ==========================================

hybrid_config = {
    "als_weight": 0.45,
    "text_weight": 0.45,
    "image_weight": 0.10,
    "top_k": 10
}

with open(
    SAVE_DIR / "hybrid_config.pkl",
    "wb"
) as f:

    pickle.dump(
        hybrid_config,
        f
    )

print("✅ Hybrid configuration saved")


# ==========================================
# FINAL
# ==========================================

print("\n========================================")
print("FINAL MODEL COMPONENTS SAVED")
print("========================================")

print(
    "Location:",
    SAVE_DIR
)

print("\nFiles saved:")

for file in sorted(SAVE_DIR.iterdir()):

    print(
        " -",
        file.name
    )

print("\n✅ SAVE COMPLETE!")

===== SAVING FINAL MODEL COMPONENTS =====
✅ ALS model saved
✅ Text embeddings saved
✅ Image index information saved
✅ Product data saved


NameError: name 'user_to_index' is not defined

In [75]:
# ==========================================
# INSPECT SAVED ALS FILE DIRECTLY
# ==========================================

import numpy as np

ALS_MODEL_PATH = (
    FINAL_MODEL_DIR / "als_model_v2.npz"
)

print("===== INSPECTING SAVED ALS FILE =====")

with np.load(
    ALS_MODEL_PATH,
    allow_pickle=False
) as data:

    print("Saved arrays:")

    for key in data.files:

        value = data[key]

        print(
            f"  {key:25} "
            f"shape={getattr(value, 'shape', 'N/A')} "
            f"dtype={getattr(value, 'dtype', 'N/A')}"
        )

print("\n✅ ALS file inspection complete!")

===== INSPECTING SAVED ALS FILE =====
Saved arrays:
  regularization            shape=() dtype=float64
  factors                   shape=() dtype=int64
  num_threads               shape=() dtype=int64
  iterations                shape=() dtype=int64
  use_native                shape=() dtype=bool
  use_cg                    shape=() dtype=bool
  cg_steps                  shape=() dtype=int64
  calculate_training_loss   shape=() dtype=bool
  dtype                     shape=() dtype=<U7
  random_state              shape=() dtype=int64
  alpha                     shape=() dtype=float64

✅ ALS file inspection complete!


In [77]:
# ==========================================
# RESTORE ALS MODEL FROM SAVED ARRAYS
# ==========================================

from implicit.cpu.als import AlternatingLeastSquares as CPU_ALS

print("===== RESTORING ALS MODEL =====")

# ------------------------------------------
# Read saved arrays
# ------------------------------------------

with np.load(
    ALS_MODEL_PATH,
    allow_pickle=False
) as data:

    saved_user_factors = data["user_factors"]
    saved_item_factors = data["item_factors"]

    saved_factors = int(
        data["factors"]
    )

    saved_regularization = float(
        data["regularization"]
    )

    saved_iterations = int(
        data["iterations"]
    )

    saved_num_threads = int(
        data["num_threads"]
    )


# ------------------------------------------
# Create ALS model
# ------------------------------------------

loaded_als_model = CPU_ALS(
    factors=saved_factors,
    regularization=saved_regularization,
    iterations=saved_iterations,
    num_threads=saved_num_threads
)

# ------------------------------------------
# Restore learned factors
# ------------------------------------------

loaded_als_model.user_factors = (
    saved_user_factors
)

loaded_als_model.item_factors = (
    saved_item_factors
)


# ------------------------------------------
# Verify
# ------------------------------------------

print(
    "User factors:",
    loaded_als_model.user_factors.shape
)

print(
    "Item factors:",
    loaded_als_model.item_factors.shape
)

print(
    "Factors:",
    loaded_als_model.factors
)

print(
    "Regularization:",
    loaded_als_model.regularization
)

print(
    "Iterations:",
    loaded_als_model.iterations
)


# ------------------------------------------
# Final verification
# ------------------------------------------

if (
    loaded_als_model.user_factors.shape
    == (2425617, 64)
    and
    loaded_als_model.item_factors.shape
    == (29489, 64)
):

    print(
        "\n✅ ALS MODEL RESTORED SUCCESSFULLY!"
    )

else:

    print(
        "\n❌ ALS MODEL DIMENSIONS ARE WRONG!"
    )

===== RESTORING ALS MODEL =====


KeyError: 'user_factors is not a file in the archive'

In [78]:
# ==========================================
# LOAD TEXT + PRODUCT COMPONENTS
# ==========================================

print("===== LOADING TEXT & PRODUCT COMPONENTS =====")

# ------------------------------------------
# Product data
# ------------------------------------------

products_df = pd.read_parquet(
    FINAL_MODEL_DIR / "products.parquet"
)

# ------------------------------------------
# Text embeddings
# ------------------------------------------

text_embeddings = np.load(
    FINAL_MODEL_DIR / "text_embeddings.npy"
)

text_norm_all = np.load(
    FINAL_MODEL_DIR / "text_norm_all.npy"
)

text_als = np.load(
    FINAL_MODEL_DIR / "text_als.npy"
)

# ------------------------------------------
# Product mappings
# ------------------------------------------

with open(
    FINAL_MODEL_DIR / "user_to_index.pkl",
    "rb"
) as f:
    user_to_index = pickle.load(f)

with open(
    FINAL_MODEL_DIR / "product_to_index.pkl",
    "rb"
) as f:
    product_to_index = pickle.load(f)

product_ids_array = np.load(
    FINAL_MODEL_DIR / "product_ids_array.npy",
    allow_pickle=True
)

# ------------------------------------------
# Catalog mappings
# ------------------------------------------

catalog_asins = np.load(
    FINAL_MODEL_DIR / "catalog_asins.npy",
    allow_pickle=True
)

with open(
    FINAL_MODEL_DIR / "catalog_asin_to_index.pkl",
    "rb"
) as f:
    catalog_asin_to_index = pickle.load(f)

# ------------------------------------------
# Image lookup
# ------------------------------------------

with open(
    FINAL_MODEL_DIR / "image_full_index_to_row.pkl",
    "rb"
) as f:
    image_full_index_to_row = pickle.load(f)

# ------------------------------------------
# Hybrid configuration
# ------------------------------------------

with open(
    FINAL_MODEL_DIR / "hybrid_config.pkl",
    "rb"
) as f:
    hybrid_config = pickle.load(f)


# ==========================================
# VERIFY
# ==========================================

print("\n===== COMPONENTS LOADED =====")

print(
    "Products:",
    products_df.shape
)

print(
    "Text embeddings:",
    text_embeddings.shape
)

print(
    "Text normalized:",
    text_norm_all.shape
)

print(
    "Text ALS:",
    text_als.shape
)

print(
    "Users:",
    len(user_to_index)
)

print(
    "ALS products:",
    len(product_ids_array)
)

print(
    "Catalog products:",
    len(catalog_asins)
)

print(
    "Hybrid config:",
    hybrid_config
)

print(
    "\n✅ TEXT & PRODUCT COMPONENTS LOADED!"
)

===== LOADING TEXT & PRODUCT COMPONENTS =====


EOFError: Ran out of input

In [79]:
# ==========================================
# LOAD IMAGE EMBEDDINGS
# ==========================================

print("===== LOADING IMAGE EMBEDDINGS =====")

IMAGE_EMBEDDING_FILE = Path(
    r"E:\FurnitureRecommendationAI_ImageEmbeddings\image_embeddings.dat"
)

# Check file
if not IMAGE_EMBEDDING_FILE.exists():
    raise FileNotFoundError(
        f"Image embedding file not found:\n"
        f"{IMAGE_EMBEDDING_FILE}"
    )

print(
    "File found:",
    IMAGE_EMBEDDING_FILE
)

# ------------------------------------------
# Memory-map the embeddings
# ------------------------------------------

image_embeddings = np.memmap(
    IMAGE_EMBEDDING_FILE,
    dtype=np.float32,
    mode="r",
    shape=(30000, 2048)
)

print(
    "\nImage embeddings shape:",
    image_embeddings.shape
)

# ------------------------------------------
# Check successful image rows
# ------------------------------------------

successful_indices_path = (
    FINAL_MODEL_DIR / "successful_indices.npy"
)

if successful_indices_path.exists():

    successful_indices = np.load(
        successful_indices_path
    )

    print(
        "Successful image indices:",
        len(successful_indices)
    )

else:

    print(
        "⚠️ successful_indices.npy not found"
    )


# ==========================================
# VERIFY SAMPLE
# ==========================================

sample_index = int(
    successful_indices[0]
)

sample_embedding = image_embeddings[
    sample_index
]

print(
    "\nSample product index:",
    sample_index
)

print(
    "Sample embedding shape:",
    sample_embedding.shape
)

print(
    "Embedding norm:",
    np.linalg.norm(
        sample_embedding
    )
)

print(
    "\nFirst 10 values:",
    sample_embedding[:10]
)

print(
    "\n✅ IMAGE EMBEDDINGS LOADED SUCCESSFULLY!"
)

===== LOADING IMAGE EMBEDDINGS =====
File found: E:\FurnitureRecommendationAI_ImageEmbeddings\image_embeddings.dat

Image embeddings shape: (30000, 2048)
Successful image indices: 28203

Sample product index: 0
Sample embedding shape: (2048,)
Embedding norm: 12.051238

First 10 values: [0.         0.         0.21819103 0.         0.         0.
 0.05424333 0.07789847 0.         0.        ]

✅ IMAGE EMBEDDINGS LOADED SUCCESSFULLY!


In [80]:
# ==========================================
# COMPLETE RESTORED MODEL TEST
# ==========================================

print("===== COMPLETE RESTORED MODEL TEST =====")

test_user = "AE2222VGHFD37G7NOCWXASYY3ZDQ"

# ------------------------------------------
# Check user
# ------------------------------------------

if test_user not in user_to_index:

    print("❌ Test user not found!")

else:

    print("✅ Test user found")
    print("User:", test_user)

    # --------------------------------------
    # ALS recommendation check
    # --------------------------------------

    user_idx = user_to_index[test_user]

    als_scores = (
        loaded_als_model.user_factors[user_idx]
        @
        loaded_als_model.item_factors.T
    )

    print(
        "\nALS scores generated:",
        als_scores.shape
    )

    # --------------------------------------
    # Basic score check
    # --------------------------------------

    print(
        "Maximum ALS score:",
        float(als_scores.max())
    )

    print(
        "Minimum ALS score:",
        float(als_scores.min())
    )

    # --------------------------------------
    # Top 10 ALS products
    # --------------------------------------

    top_indices = np.argsort(
        -als_scores
    )[:10]

    print("\n===== TOP 10 ALS PRODUCTS =====")

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        asin = product_ids_array[idx]

        print(
            f"{rank}. {asin}"
            f" | Score: {als_scores[idx]:.6f}"
        )

    print(
        "\n✅ RESTORED ALS MODEL IS WORKING!"
    )

===== COMPLETE RESTORED MODEL TEST =====


NameError: name 'user_to_index' is not defined

In [81]:
# ==========================================
# COMPLETE RESTORED HYBRID MODEL TEST
# ==========================================

print("===== COMPLETE RESTORED HYBRID TEST =====")

test_user = "AE2222VGHFD37G7NOCWXASYY3ZDQ"

# ------------------------------------------
# Check user
# ------------------------------------------

if test_user not in user_to_index:

    print("❌ User not found!")

else:

    print("✅ User found:", test_user)

    user_idx = user_to_index[test_user]

    # --------------------------------------
    # 1. ALS SCORES
    # --------------------------------------

    als_scores = (
        loaded_als_model.user_factors[user_idx]
        @
        loaded_als_model.item_factors.T
    )

    # --------------------------------------
    # 2. USER TEXT PROFILE
    # --------------------------------------

    train_products = train_positive_lookup.get(
        test_user,
        set()
    )

    text_indices = [
        catalog_asin_to_index[asin]
        for asin in train_products
        if asin in catalog_asin_to_index
    ]

    if len(text_indices) > 0:

        user_text_profile = np.mean(
            text_norm_all[text_indices],
            axis=0
        )

        user_text_profile /= (
            np.linalg.norm(
                user_text_profile
            ) + 1e-8
        )

        text_scores = (
            text_als
            @
            user_text_profile
        )

    else:

        text_scores = np.zeros(
            len(als_asins),
            dtype=np.float32
        )

    # --------------------------------------
    # 3. IMAGE PROFILE
    # --------------------------------------

    image_profile = create_user_image_profile(
        test_user
    )

    image_scores = np.zeros(
        len(als_asins),
        dtype=np.float32
    )

    if image_profile is not None:

        image_scores_successful = (
            image_embeddings[
                successful_indices
            ]
            @
            image_profile
        )

        for als_idx, full_idx in enumerate(
            als_full_indices
        ):

            image_row = (
                image_full_index_to_row.get(
                    int(full_idx)
                )
            )

            if image_row is not None:

                image_scores[als_idx] = (
                    image_scores_successful[
                        image_row
                    ]
                )

    # --------------------------------------
    # 4. NORMALIZATION
    # --------------------------------------

    def minmax(x):

        minimum = x.min()
        maximum = x.max()

        if maximum == minimum:
            return np.zeros_like(x)

        return (
            (x - minimum)
            /
            (maximum - minimum)
        )

    als_norm = minmax(als_scores)

    text_norm_scores = minmax(
        text_scores
    )

    image_norm_scores = minmax(
        image_scores
    )

    # --------------------------------------
    # 5. HYBRID SCORE
    # --------------------------------------

    als_weight = hybrid_config[
        "als_weight"
    ]

    text_weight = hybrid_config[
        "text_weight"
    ]

    image_weight = hybrid_config[
        "image_weight"
    ]

    hybrid_scores = (
        als_weight * als_norm
        +
        text_weight * text_norm_scores
        +
        image_weight * image_norm_scores
    )

    # --------------------------------------
    # 6. REMOVE SEEN PRODUCTS
    # --------------------------------------

    hybrid_scores = hybrid_scores.copy()

    for idx, asin in enumerate(als_asins):

        if asin in train_products:

            hybrid_scores[idx] = -np.inf

    # --------------------------------------
    # 7. TOP 10
    # --------------------------------------

    top_indices = np.argsort(
        -hybrid_scores
    )[:10]

    # --------------------------------------
    # 8. DISPLAY RESULTS
    # --------------------------------------

    print("\n===== RESTORED HYBRID TOP 10 =====")

    for rank, idx in enumerate(
        top_indices,
        start=1
    ):

        asin = als_asins[idx]

        product_rows = products_df[
            products_df["parent_asin"] == asin
        ]

        if len(product_rows) > 0:

            title = product_rows.iloc[0].get(
                "title",
                "Unknown"
            )

        else:

            title = "Unknown"

        print(
            f"{rank}. {title}"
        )

        print(
            f"   ASIN: {asin}"
        )

        print(
            f"   ALS: {als_norm[idx]:.4f}"
        )

        print(
            f"   Text: {text_norm_scores[idx]:.4f}"
        )

        print(
            f"   Image: {image_norm_scores[idx]:.4f}"
        )

        print(
            f"   Hybrid: {hybrid_scores[idx]:.4f}"
        )

    print(
        "\n✅ COMPLETE RESTORED HYBRID TEST PASSED!"
    )

===== COMPLETE RESTORED HYBRID TEST =====


NameError: name 'user_to_index' is not defined

In [82]:
# ==========================================
# FINAL MODEL CONSISTENCY CHECK
# ==========================================

print("===== FINAL MODEL CONSISTENCY CHECK =====")

test_user = "AE2222VGHFD37G7NOCWXASYY3ZDQ"

# Generate recommendations from the restored model
restored_results = []

user_idx = user_to_index[test_user]

# ------------------------------------------
# ALS
# ------------------------------------------

als_scores = (
    loaded_als_model.user_factors[user_idx]
    @ loaded_als_model.item_factors.T
)

# ------------------------------------------
# Training products
# ------------------------------------------

train_products = train_positive_lookup.get(
    test_user,
    set()
)

# ------------------------------------------
# TEXT
# ------------------------------------------

text_indices = [
    catalog_asin_to_index[asin]
    for asin in train_products
    if asin in catalog_asin_to_index
]

user_text_profile = np.mean(
    text_norm_all[text_indices],
    axis=0
)

user_text_profile /= (
    np.linalg.norm(user_text_profile) + 1e-8
)

text_scores = text_als @ user_text_profile


# ------------------------------------------
# IMAGE
# ------------------------------------------

image_profile = create_user_image_profile(
    test_user
)

image_scores = np.zeros(
    len(als_asins),
    dtype=np.float32
)

if image_profile is not None:

    image_scores_successful = (
        image_embeddings[successful_indices]
        @ image_profile
    )

    for als_idx, full_idx in enumerate(
        als_full_indices
    ):

        image_row = image_full_index_to_row.get(
            int(full_idx)
        )

        if image_row is not None:

            image_scores[als_idx] = (
                image_scores_successful[image_row]
            )


# ------------------------------------------
# NORMALIZATION
# ------------------------------------------

def minmax(x):

    minimum = x.min()
    maximum = x.max()

    if maximum == minimum:
        return np.zeros_like(x)

    return (
        (x - minimum)
        / (maximum - minimum)
    )


als_norm = minmax(als_scores)
text_norm_scores = minmax(text_scores)
image_norm_scores = minmax(image_scores)


# ------------------------------------------
# HYBRID
# ------------------------------------------

hybrid_scores = (
    0.45 * als_norm
    + 0.45 * text_norm_scores
    + 0.10 * image_norm_scores
)


# ------------------------------------------
# REMOVE SEEN PRODUCTS
# ------------------------------------------

hybrid_scores = hybrid_scores.copy()

for idx, asin in enumerate(als_asins):

    if asin in train_products:
        hybrid_scores[idx] = -np.inf


# ------------------------------------------
# TOP 10
# ------------------------------------------

top_indices = np.argsort(
    -hybrid_scores
)[:10]


# ------------------------------------------
# CREATE TABLE
# ------------------------------------------

for rank, idx in enumerate(
    top_indices,
    start=1
):

    asin = als_asins[idx]

    product_rows = products_df[
        products_df["parent_asin"] == asin
    ]

    if len(product_rows) > 0:

        title = product_rows.iloc[0]["title"]

    else:

        title = "Unknown"

    restored_results.append({
        "Rank": rank,
        "ASIN": asin,
        "Title": title,
        "ALS": round(float(als_norm[idx]), 4),
        "Text": round(float(text_norm_scores[idx]), 4),
        "Image": round(float(image_norm_scores[idx]), 4),
        "Hybrid": round(float(hybrid_scores[idx]), 4)
    })


# ------------------------------------------
# DISPLAY
# ------------------------------------------

restored_results_df = pd.DataFrame(
    restored_results
)

print("\n===== FINAL RESTORED RECOMMENDATIONS =====")

display(restored_results_df)

print(
    "\n✅ FINAL MODEL CONSISTENCY CHECK COMPLETE!"
)

===== FINAL MODEL CONSISTENCY CHECK =====


NameError: name 'user_to_index' is not defined

In [83]:
# ==========================================
# FIND TRAINING SPARSE MATRIX
# ==========================================

import scipy.sparse as sp

print("===== SEARCHING FOR SPARSE MATRICES =====")

found = []

for name, value in globals().items():

    if sp.issparse(value):

        print(
            f"Variable: {name}"
        )

        print(
            f"Shape: {value.shape}"
        )

        print(
            f"Non-zero: {value.nnz}"
        )

        print()

        if (
            value.shape == (2425617, 29489)
            and value.nnz == 8797821
        ):

            found.append(name)


print(
    "===== MATCHING MATRICES ====="
)

print(found)

if len(found) == 0:

    print(
        "❌ Exact training matrix was not found."
    )

else:

    print(
        "✅ Training matrix found:",
        found
    )

===== SEARCHING FOR SPARSE MATRICES =====
===== MATCHING MATRICES =====
[]
❌ Exact training matrix was not found.


In [84]:
# ==========================================
# SAVE TRAINING INTERACTION MATRIX
# ==========================================

from scipy.sparse import save_npz
from pathlib import Path

print("===== SAVING TRAINING INTERACTION MATRIX =====")

FINAL_MODEL_DIR = Path(
    r"E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL"
)

FINAL_MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

# Use the matrix we found
matrix_to_save = user_item_matrix

print(
    "Matrix shape:",
    matrix_to_save.shape
)

print(
    "Non-zero interactions:",
    matrix_to_save.nnz
)

# Save
matrix_path = (
    FINAL_MODEL_DIR
    / "train_interaction_matrix.npz"
)

save_npz(
    matrix_path,
    matrix_to_save
)

print(
    "\nFile saved to:"
)

print(matrix_path)

print(
    "\n✅ TRAINING INTERACTION MATRIX SAVED!"
)

===== SAVING TRAINING INTERACTION MATRIX =====


NameError: name 'user_item_matrix' is not defined

In [85]:
print("===== PRODUCT COLUMNS =====")

print(products_df.columns.tolist())

print("\n===== FIRST 5 PRODUCTS =====")

display(products_df.head())

===== PRODUCT COLUMNS =====
['parent_asin', 'title', 'image_url', 'price', 'average_rating', 'rating_number', 'main_category', 'description', 'categories']

===== FIRST 5 PRODUCTS =====


,parent_asin,title,image_url,price,average_rating,rating_number,main_category,description,categories
0,B0C4CPL2RV,"Walensee Large Bathroom Rug (24 x 60, Black) E...",https://m.media-amazon.com/images/I/61PYg8i0C5...,41.99,4.3,31438,Amazon Home,[],"[Home & Kitchen, Bath, Bath Rugs]"
1,B09L626MHB,CHILLOUT LIFE 20 oz Stainless Steel Tumbler wi...,https://m.media-amazon.com/images/I/71hTlPC2lk...,13.49,4.7,18386,Amazon Home,[],"[Home & Kitchen, Kitchen & Dining, Dining & En..."
2,B0BPJ69P2H,H.VERSAILTEX Blackout Curtains 84 Inch Length ...,https://m.media-amazon.com/images/I/71z4l+ypmB...,34.99,4.7,2696,Amazon Home,[],"[Home & Kitchen, Home Décor Products, Window T..."
3,B00JIKM0UQ,Best Home Fashion Wide Width Thermal Insulated...,https://m.media-amazon.com/images/I/81ImB8Gp2+...,24.99,4.5,614,Amazon Home,[Best Home Fashion introduces the new Blackout...,"[Home & Kitchen, Home Décor Products, Window T..."
4,B081J8DH1K,Bedsure Linen Duvet Cover Queen Linen Cotton B...,https://m.media-amazon.com/images/I/81DZ1lzwLB...,50.99,4.5,2845,Amazon Home,[],"[Home & Kitchen, Bedding, Duvet Covers & Sets,..."


In [86]:
print("===== DATASET FILES =====")

from pathlib import Path

for path in Path(
    r"E:\FurnitureRecommendationAI_ImageEmbeddings"
).rglob("*"):

    if path.is_file():

        print(path)

===== DATASET FILES =====
E:\FurnitureRecommendationAI_ImageEmbeddings\image_embeddings.dat
E:\FurnitureRecommendationAI_ImageEmbeddings\image_status.npy
E:\FurnitureRecommendationAI_ImageEmbeddings\ALS_V2\als_model_v2.npz
E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL\als_model_v2.npz
E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL\catalog_asins.npy
E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL\catalog_asin_to_index.pkl
E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL\hybrid_config.pkl
E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL\image_full_index_to_row.pkl
E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL\products.parquet
E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL\product_ids_array.npy
E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL\product_metadata.parquet
E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL\product_to_index.pkl
E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL\successful_indi

In [87]:
print("===== TRAIN DATA COLUMNS =====")
print(train_df.columns.tolist())

print("\n===== TRAIN DATA SAMPLE =====")
display(train_df.head())

===== TRAIN DATA COLUMNS =====


NameError: name 'train_df' is not defined

In [88]:
print("===== ALL DATAFRAME VARIABLES =====")

for name, value in globals().items():

    if hasattr(value, "columns"):

        try:
            print(
                name,
                "→",
                list(value.columns)
            )
        except:
            pass

===== ALL DATAFRAME VARIABLES =====
products_df → ['parent_asin', 'title', 'image_url', 'price', 'average_rating', 'rating_number', 'main_category', 'description', 'categories']
products_text_df → ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'store', 'categories', 'details', 'parent_asin', 'bought_together', 'subtitle', 'author', 'image_count', 'image_url', 'product_text']
product_metadata → ['parent_asin', 'title', 'image_url', 'price', 'average_rating', 'rating_number', 'main_category', 'description', 'categories']
products → ['parent_asin', 'title', 'image_url', 'price', 'average_rating', 'rating_number', 'main_category', 'description', 'categories', 'review_score']


In [89]:
# ==========================================
# SAVE PRODUCT SEARCH METADATA
# ==========================================

from pathlib import Path

FINAL_MODEL_DIR = Path(
    r"E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL"
)

metadata_columns = [
    "parent_asin",
    "title",
    "image_url",
    "price",
    "average_rating",
    "rating_number",
    "main_category",
    "description",
    "categories"
]

product_metadata = products_text_df[
    metadata_columns
].copy()

metadata_path = (
    FINAL_MODEL_DIR /
    "product_metadata.parquet"
)

product_metadata.to_parquet(
    metadata_path,
    index=False
)

print(
    "===== PRODUCT SEARCH METADATA SAVED ====="
)

print(
    "Shape:",
    product_metadata.shape
)

print(
    "Columns:",
    product_metadata.columns.tolist()
)

print(
    "File:",
    metadata_path
)

print(
    "\n✅ PRODUCT SEARCH METADATA SAVED!"
)

===== PRODUCT SEARCH METADATA SAVED =====
Shape: (30000, 9)
Columns: ['parent_asin', 'title', 'image_url', 'price', 'average_rating', 'rating_number', 'main_category', 'description', 'categories']
File: E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL\product_metadata.parquet

✅ PRODUCT SEARCH METADATA SAVED!


In [90]:
# ==========================================
# VERIFY PRODUCT SEARCH METADATA
# ==========================================

from pathlib import Path
import pandas as pd

FINAL_MODEL_DIR = Path(
    r"E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL"
)

metadata_path = FINAL_MODEL_DIR / "product_metadata.parquet"

print("===== LOADING PRODUCT SEARCH METADATA =====")

product_metadata = pd.read_parquet(metadata_path)

print(
    "Shape:",
    product_metadata.shape
)

print(
    "\nColumns:"
)

print(
    product_metadata.columns.tolist()
)

print(
    "\n===== FIRST 5 PRODUCTS ====="
)

display(
    product_metadata.head()
)

print(
    "\n===== PRICE DATA ====="
)

print(
    product_metadata["price"].head(10)
)

print(
    "\n===== RATING DATA ====="
)

print(
    product_metadata[
        [
            "average_rating",
            "rating_number"
        ]
    ].head(10)
)

print(
    "\n✅ PRODUCT SEARCH METADATA LOADED SUCCESSFULLY!"
)

===== LOADING PRODUCT SEARCH METADATA =====
Shape: (30000, 9)

Columns:
['parent_asin', 'title', 'image_url', 'price', 'average_rating', 'rating_number', 'main_category', 'description', 'categories']

===== FIRST 5 PRODUCTS =====


,parent_asin,title,image_url,price,average_rating,rating_number,main_category,description,categories
0,B0C4CPL2RV,"Walensee Large Bathroom Rug (24 x 60, Black) E...",https://m.media-amazon.com/images/I/61PYg8i0C5...,41.99,4.3,31438,Amazon Home,[],"[Home & Kitchen, Bath, Bath Rugs]"
1,B09L626MHB,CHILLOUT LIFE 20 oz Stainless Steel Tumbler wi...,https://m.media-amazon.com/images/I/71hTlPC2lk...,13.49,4.7,18386,Amazon Home,[],"[Home & Kitchen, Kitchen & Dining, Dining & En..."
2,B0BPJ69P2H,H.VERSAILTEX Blackout Curtains 84 Inch Length ...,https://m.media-amazon.com/images/I/71z4l+ypmB...,34.99,4.7,2696,Amazon Home,[],"[Home & Kitchen, Home Décor Products, Window T..."
3,B00JIKM0UQ,Best Home Fashion Wide Width Thermal Insulated...,https://m.media-amazon.com/images/I/81ImB8Gp2+...,24.99,4.5,614,Amazon Home,[Best Home Fashion introduces the new Blackout...,"[Home & Kitchen, Home Décor Products, Window T..."
4,B081J8DH1K,Bedsure Linen Duvet Cover Queen Linen Cotton B...,https://m.media-amazon.com/images/I/81DZ1lzwLB...,50.99,4.5,2845,Amazon Home,[],"[Home & Kitchen, Bedding, Duvet Covers & Sets,..."



===== PRICE DATA =====
0    41.99
1    13.49
2    34.99
3    24.99
4    50.99
5    11.21
6    15.99
7    14.99
8    25.99
9    56.78
Name: price, dtype: str

===== RATING DATA =====
   average_rating  rating_number
0             4.3          31438
1             4.7          18386
2             4.7           2696
3             4.5            614
4             4.5           2845
5             4.6           6209
6             4.6           7236
7             4.7           2403
8             4.6           8143
9             4.3           2386

✅ PRODUCT SEARCH METADATA LOADED SUCCESSFULLY!


In [91]:
import numpy as np

TEXT_EMBEDDINGS_PATH = r"E:\FurnitureRecommendationAI_ImageEmbeddings\FINAL_MODEL\text_embeddings.npy"

text_embeddings = np.load(TEXT_EMBEDDINGS_PATH)

print("===== TEXT EMBEDDINGS =====")
print("Shape:", text_embeddings.shape)
print("Dimension:", text_embeddings.shape[1])

===== TEXT EMBEDDINGS =====
Shape: (30000, 384)
Dimension: 384


In [92]:
import os

print("===== SEARCHING FOR TEXT MODEL =====")

for name, value in globals().items():
    if "model" in name.lower():
        print(name, "->", type(value))

===== SEARCHING FOR TEXT MODEL =====
als_model_v2 -> <class 'implicit.cpu.als.AlternatingLeastSquares'>
model_configs -> <class 'dict'>
model_name -> <class 'str'>
FINAL_MODEL_DIR -> <class 'pathlib.WindowsPath'>
AutoModel -> <class 'type'>
model_path -> <class 'str'>
text_model -> <class 'transformers.models.bert.modeling_bert.BertModel'>
als_model_path -> <class 'pathlib.WindowsPath'>
ALS_MODEL_PATH -> <class 'pathlib.WindowsPath'>


In [93]:
print("===== CURRENT MODEL CHECK =====")

print("text_model exists:", "text_model" in globals())
print("tokenizer exists:", "tokenizer" in globals())
print("torch exists:", "torch" in globals())

if "text_model" in globals():
    print("text_model type:", type(text_model))

if "tokenizer" in globals():
    print("tokenizer type:", type(tokenizer))

===== CURRENT MODEL CHECK =====
text_model exists: True
tokenizer exists: True
torch exists: True
text_model type: <class 'transformers.models.bert.modeling_bert.BertModel'>
tokenizer type: <class 'transformers.models.bert.tokenization_bert.BertTokenizer'>


In [94]:
def create_query_embedding(query):

    text_model.eval()

    inputs = tokenizer(
        [query],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {
        key: value.to("cuda")
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = text_model(**inputs)

        token_embeddings = outputs.last_hidden_state

        attention_mask = inputs["attention_mask"]

        mask = attention_mask.unsqueeze(-1).expand(
            token_embeddings.size()
        ).float()

        sum_embeddings = torch.sum(
            token_embeddings * mask,
            dim=1
        )

        sum_mask = torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

        embedding = sum_embeddings / sum_mask

        embedding = torch.nn.functional.normalize(
            embedding,
            p=2,
            dim=1
        )

    return embedding.cpu().numpy()[0]

In [95]:
import torch

print("================================")
print("PYTORCH TEST")
print("================================")
print("PyTorch version :", torch.__version__)
print("CUDA available  :", torch.cuda.is_available())
print("CUDA version    :", torch.version.cuda)

if torch.cuda.is_available():
    print("GPU             :", torch.cuda.get_device_name(0))
    print("GPU count       :", torch.cuda.device_count())
else:
    print("GPU             : Not available")

print("================================")

PYTORCH TEST
PyTorch version : 2.9.1+cu126
CUDA available  : True
CUDA version    : 12.6
GPU             : NVIDIA GeForce RTX 4050 Laptop GPU
GPU count       : 1


In [96]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel

print("Loading MiniLM...")

model_path = r"C:\Users\amjad\FurnitureRecommendationAI_Models\all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    local_files_only=True
)

text_model = AutoModel.from_pretrained(
    model_path,
    local_files_only=True
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

text_model = text_model.to(device)
text_model.eval()

print("================================")
print("MINILM LOADED")
print("================================")
print("Device:", device)
print("Model:", model_path)

Loading MiniLM...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10587.00it/s]

MINILM LOADED
Device: cuda
Model: C:\Users\amjad\FurnitureRecommendationAI_Models\all-MiniLM-L6-v2


In [97]:
def create_query_embedding(query):

    text_model.eval()

    inputs = tokenizer(
        [query],
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    inputs = {
        key: value.to(device)
        for key, value in inputs.items()
    }

    with torch.no_grad():

        outputs = text_model(**inputs)

        token_embeddings = outputs.last_hidden_state

        attention_mask = inputs["attention_mask"]

        mask = attention_mask.unsqueeze(-1).expand(
            token_embeddings.size()
        ).float()

        sum_embeddings = torch.sum(
            token_embeddings * mask,
            dim=1
        )

        sum_mask = torch.clamp(
            mask.sum(dim=1),
            min=1e-9
        )

        embedding = sum_embeddings / sum_mask

        # Same normalization used for your
        # original 30,000 product embeddings
        embedding = torch.nn.functional.normalize(
            embedding,
            p=2,
            dim=1
        )

    return embedding.cpu().numpy()[0]


print("Query embedding function ready!")

Query embedding function ready!


In [98]:
query_embedding = create_query_embedding(
    "kitchen equipment"
)

print("================================")
print("QUERY EMBEDDING TEST")
print("================================")
print("Shape:", query_embedding.shape)
print("Norm:", np.linalg.norm(query_embedding))

QUERY EMBEDDING TEST
Shape: (384,)
Norm: 1.0


In [99]:
TEXT_EMBEDDINGS_PATH = (
    r"E:\FurnitureRecommendationAI_ImageEmbeddings"
    r"\FINAL_MODEL\text_embeddings.npy"
)

text_embeddings = np.load(TEXT_EMBEDDINGS_PATH)

print("================================")
print("PRODUCT EMBEDDINGS")
print("================================")
print("Shape:", text_embeddings.shape)

PRODUCT EMBEDDINGS
Shape: (30000, 384)


In [100]:
import pandas as pd
import numpy as np

# ==========================================
# LOAD PRODUCT METADATA
# ==========================================

METADATA_PATH = (
    r"E:\FurnitureRecommendationAI_ImageEmbeddings"
    r"\FINAL_MODEL\product_metadata.parquet"
)

products_df = pd.read_parquet(METADATA_PATH)

print("Products loaded:", products_df.shape)


# ==========================================
# LOAD EXISTING TEXT EMBEDDINGS
# ==========================================

TEXT_EMBEDDINGS_PATH = (
    r"E:\FurnitureRecommendationAI_ImageEmbeddings"
    r"\FINAL_MODEL\text_embeddings.npy"
)

text_embeddings = np.load(TEXT_EMBEDDINGS_PATH)

print("Text embeddings:", text_embeddings.shape)


# ==========================================
# NORMALIZE PRODUCT EMBEDDINGS
# ==========================================

text_embeddings_norm = text_embeddings / np.maximum(
    np.linalg.norm(
        text_embeddings,
        axis=1,
        keepdims=True
    ),
    1e-12
)


# ==========================================
# CALCULATE SEMANTIC SIMILARITY
# ==========================================

similarities = text_embeddings_norm @ query_embedding


# ==========================================
# GET TOP 20 PRODUCTS
# ==========================================

top_indices = np.argsort(similarities)[::-1][:20]


# ==========================================
# DISPLAY RESULTS
# ==========================================

print("\n================================")
print("TOP SEMANTIC RESULTS")
print("================================")

for rank, idx in enumerate(top_indices, start=1):

    title = str(products_df.iloc[idx]["title"])

    print(
        f"{rank:2d}. "
        f"{title[:100]} "
        f"| similarity={similarities[idx]:.4f}"
    )

Products loaded: (30000, 9)
Text embeddings: (30000, 384)

TOP SEMANTIC RESULTS
 1. KitchenAid Classic Tool and Gadget Set, 15-Piece, Black | similarity=0.5897
 2. SHARDOR Hand Mixer, 350W Handheld Mixer with Storage Case 5-Speed Plus Turbo Hand Mixer Electric Wit | similarity=0.5879
 3. Kitchen Utensils Set, Cooking Utensil Sets Kitchen Gadgets, Pots and Pans set Nonstick and Heat Resi | similarity=0.5872
 4. Hand Mixer Electric Mixing Bowls Set, 5 Speeds Handheld Mixer with 5 Nesting Stainless Steel Mixing  | similarity=0.5825
 5. Joseph Joseph Uni-Tool 5-In-1 Utensil Slotted Solid Spoon Spatula Turner Slicer Kitchen All-In One S | similarity=0.5686
 6. DmofwHi Hand Mixer Electric,5-Speed Mixer Electric Handheld with 6 Stainless Steel Accessories and S | similarity=0.5666
 7. Breville Sous Chef 12 Cup Food Processor, Silver, BFP660SIL | similarity=0.5662
 8. 25-Piece Stainless Steel Kitchen Utensil Set | Non-Stick Cooking Gadgets and Tools Kit | Durable Dis | similarity=0.5653
 9. SM

In [101]:
import pandas as pd
import numpy as np


# ============================================================
# 1. PREPARE PRODUCT DATA
# ============================================================

products = products_df.copy()

# Convert important columns to numeric
products["price"] = pd.to_numeric(
    products["price"],
    errors="coerce"
)

products["average_rating"] = pd.to_numeric(
    products["average_rating"],
    errors="coerce"
)

products["rating_number"] = pd.to_numeric(
    products["rating_number"],
    errors="coerce"
)

# Missing values
products["price"] = products["price"].fillna(np.inf)
products["average_rating"] = products["average_rating"].fillna(0)
products["rating_number"] = products["rating_number"].fillna(0)


# ============================================================
# 2. CALCULATE BAYESIAN REVIEW SCORE
# ============================================================

global_mean = products.loc[
    products["average_rating"] > 0,
    "average_rating"
].mean()

minimum_reviews = 100

products["review_score"] = (
    (
        products["rating_number"] /
        (products["rating_number"] + minimum_reviews)
    )
    * products["average_rating"]
    +
    (
        minimum_reviews /
        (products["rating_number"] + minimum_reviews)
    )
    * global_mean
)

print("Product data prepared!")
print("Global rating:", round(global_mean, 3))


# ============================================================
# 3. FINAL PRODUCT SEARCH FUNCTION
# ============================================================

def search_products(
    query,
    max_price=None,
    min_rating=0,
    top_k=10
):

    # --------------------------------------------------------
    # Create query embedding
    # --------------------------------------------------------

    query_embedding = create_query_embedding(query)

    # --------------------------------------------------------
    # Semantic similarity
    # --------------------------------------------------------

    similarities = text_embeddings_norm @ query_embedding

    # --------------------------------------------------------
    # Create result dataframe
    # --------------------------------------------------------

    results = products.copy()

    results["semantic_score"] = similarities

    # --------------------------------------------------------
    # PRICE FILTER
    # --------------------------------------------------------

    if max_price is not None:

        results = results[
            results["price"] <= max_price
        ]

    # --------------------------------------------------------
    # RATING FILTER
    # --------------------------------------------------------

    if min_rating > 0:

        results = results[
            results["average_rating"] >= min_rating
        ]

    # --------------------------------------------------------
    # RANKING
    # --------------------------------------------------------
    #
    # Semantic relevance is the main signal.
    # Review score helps highly-rated products with
    # many reviews rank above products with very few reviews.
    #

    results["final_score"] = (
        0.75 * results["semantic_score"]
        +
        0.25 * (
            results["review_score"] / 5.0
        )
    )

    # --------------------------------------------------------
    # Sort
    # --------------------------------------------------------

    results = results.sort_values(
        "final_score",
        ascending=False
    )

    # --------------------------------------------------------
    # Return top products
    # --------------------------------------------------------

    return results.head(top_k).reset_index(drop=True)

Product data prepared!
Global rating: 4.465


In [102]:
results = search_products(
    query="kitchen equipment",
    max_price=100,
    min_rating=4.0,
    top_k=10
)

print("==========================================")
print("KITCHEN EQUIPMENT — UNDER $100")
print("==========================================")

for i, row in results.iterrows():

    print(f"\n{i + 1}. {row['title']}")
    print(f"   Price: ${row['price']:.2f}")
    print(f"   Rating: {row['average_rating']:.1f}")
    print(f"   Reviews: {int(row['rating_number']):,}")
    print(f"   Semantic score: {row['semantic_score']:.4f}")
    print(f"   Final score: {row['final_score']:.4f}")

KITCHEN EQUIPMENT — UNDER $100

1. KitchenAid Classic Tool and Gadget Set, 15-Piece, Black
   Price: $35.54
   Rating: 4.7
   Reviews: 2,894
   Semantic score: 0.5897
   Final score: 0.6769

2. Kitchen Utensils Set, Cooking Utensil Sets Kitchen Gadgets, Pots and Pans set Nonstick and Heat Resistant, 24 Pcs Nylon and Stainless Steel, Spatula Set, Kitchen, Home, House, Essentials & Accessories
   Price: $18.96
   Rating: 4.5
   Reviews: 6,414
   Semantic score: 0.5872
   Final score: 0.6653

3. Hand Mixer Electric Mixing Bowls Set, 5 Speeds Handheld Mixer with 5 Nesting Stainless Steel Mixing Bowl, Measuring Cups and Spoons 200 Watt Kitchen Blender Whisk Beater Baking Supplies For Beginner
   Price: $39.99
   Rating: 4.5
   Reviews: 9,032
   Semantic score: 0.5825
   Final score: 0.6618

4. SHARDOR Hand Mixer, 350W Handheld Mixer with Storage Case 5-Speed Plus Turbo Hand Mixer Electric With 5 Stainless Steel Attachments(2 Beaters, 2 Dough Hooks and 1 Whisk), Silver
   Price: $37.98
   Ra

I found 10 products matching your search for 'kitchen equipment'.

========== BEST VALUE ==========
KitchenAid Classic Tool and Gadget Set, 15-Piece, Black

========== CHEAPEST ==========
Joseph Joseph Uni-Tool 5-In-1 Utensil Slotted Solid Spoon Spatula Turner Slicer Kitchen All-In One Space Saving, Gray 12 long

========== HIGHEST RATED ==========
VASAGLE Baker's Rack, Microwave Stand, Kitchen Storage Rack with Wire Basket, 6 Hooks, and Shelves, for Spices, Pots, and Pans, Rustic Brown and Black UKKS35X

========== MOST REVIEWED ==========
Hand Mixer Electric Mixing Bowls Set, 5 Speeds Handheld Mixer with 5 Nesting Stainless Steel Mixing Bowl, Measuring Cups and Spoons 200 Watt Kitchen Blender Whisk Beater Baking Supplies For Beginner


In [149]:
import os

print("Current working directory:")
print(os.getcwd())

Current working directory:
c:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI\notebooks


In [150]:
import sys

PROJECT_PATH = (
    r"C:\Users\amjad\OneDrive\Desktop"
    r"\Data_science_works"
    r"\FurnitureRecommendationAI"
)

if PROJECT_PATH not in sys.path:
    sys.path.insert(0, PROJECT_PATH)

print("Project path added:")
print(PROJECT_PATH)

Project path added:
C:\Users\amjad\OneDrive\Desktop\Data_science_works\FurnitureRecommendationAI


In [ ]:
import os

BACKEND_PATH = os.path.join(
    PROJECT_PATH,
    "backend"
)

print("Backend folder exists:", os.path.exists(BACKEND_PATH))

print("\nBackend files:")

if os.path.exists(BACKEND_PATH):
    print(os.listdir(BACKEND_PATH))

Backend folder exists: True

Backend files:
['model_loader.py', 'product_search.py', 'recommender.py', 'semantic_search.py', 'shopping_agent.py', 'test_model_loader.py', 'test_product_search.py', 'test_recommender.py', '__pycache__']


In [ ]:
from backend.shopping_agent import generate_buying_advice

print("Shopping agent imported successfully! ✅")

Shopping agent imported successfully! ✅


In [ ]:
advice = generate_buying_advice(
    products=results,
    query="kitchen equipment"
)

print(advice["message"])

print("\n========== BEST VALUE ==========")
print(advice["best_value"]["title"])

print("\n========== CHEAPEST ==========")
print(advice["cheapest"]["title"])

print("\n========== HIGHEST RATED ==========")
print(advice["highest_rated"]["title"])

print("\n========== MOST REVIEWED ==========")
print(advice["most_reviewed"]["title"])

I found 10 products matching your search for 'kitchen equipment'.

========== BEST VALUE ==========
KitchenAid Classic Tool and Gadget Set, 15-Piece, Black

========== CHEAPEST ==========
Joseph Joseph Uni-Tool 5-In-1 Utensil Slotted Solid Spoon Spatula Turner Slicer Kitchen All-In One Space Saving, Gray 12 long

========== HIGHEST RATED ==========
VASAGLE Baker's Rack, Microwave Stand, Kitchen Storage Rack with Wire Basket, 6 Hooks, and Shelves, for Spices, Pots, and Pans, Rustic Brown and Black UKKS35X

========== MOST REVIEWED ==========
Hand Mixer Electric Mixing Bowls Set, 5 Speeds Handheld Mixer with 5 Nesting Stainless Steel Mixing Bowl, Measuring Cups and Spoons 200 Watt Kitchen Blender Whisk Beater Baking Supplies For Beginner


In [152]:
from backend.shopping_agent import generate_buying_advice

advice = generate_buying_advice(
    products=results,
    query="kitchen equipment"
)

print(advice["message"])

print("\n========== BEST VALUE ==========")
print(
    advice["best_value"]["title"]
)

print("\n========== CHEAPEST ==========")
print(
    advice["cheapest"]["title"]
)

print("\n========== HIGHEST RATED ==========")
print(
    advice["highest_rated"]["title"]
)

print("\n========== MOST REVIEWED ==========")
print(
    advice["most_reviewed"]["title"]
)

I found 10 products matching your search for 'kitchen equipment'.

========== BEST VALUE ==========
KitchenAid Classic Tool and Gadget Set, 15-Piece, Black

========== CHEAPEST ==========
Joseph Joseph Uni-Tool 5-In-1 Utensil Slotted Solid Spoon Spatula Turner Slicer Kitchen All-In One Space Saving, Gray 12 long

========== HIGHEST RATED ==========
VASAGLE Baker's Rack, Microwave Stand, Kitchen Storage Rack with Wire Basket, 6 Hooks, and Shelves, for Spices, Pots, and Pans, Rustic Brown and Black UKKS35X

========== MOST REVIEWED ==========
Hand Mixer Electric Mixing Bowls Set, 5 Speeds Handheld Mixer with 5 Nesting Stainless Steel Mixing Bowl, Measuring Cups and Spoons 200 Watt Kitchen Blender Whisk Beater Baking Supplies For Beginner
